# Four weeks of negative results, and the finding I had to take back twice

### Playground Series S6E8, smartphone addiction

Best cross-validation **0.967925**, best public LB **0.96924**. Every logged
experiment is in the ledger at the bottom, and sixteen of them are nulls or negatives
that are written up here at the same length as the wins.

This is not a solution notebook. Nothing in it will move you up the leaderboard by
itself. It is a record of which ideas worked, which did not, and how each one was
killed cheaply enough that the next one still had time to be tried.

**The short version.** Fifteen experiments of model capacity and stochastic averaging
were worth **+0.0089 AUC** between them, and 80% of that was noticing the untuned
baseline was underfit. Not one engineered feature helped. Then one change of
*representation*, target encoding every column, was worth **+0.0029** on its own, and
it arrived after I had already written that the board was closed.

Then the interesting part. That change of representation quietly invalidated three
rejections I had already made and filed, and going back for them was worth more than
anything I did on purpose in the last two weeks.

Six things in here that I have not seen written up elsewhere for this dataset:

1. **Rank correlation between two models does not predict whether averaging them
   helps.** Measured across all 66 pairs of models I trained. This is the opposite of
   the usual advice, and it nearly cost me a day of CatBoost runs.
2. **I then over-claimed that finding, and the correction is the most useful thing
   here.** Every test behind it used an equal-weight combiner. Hold the eighteen
   models fixed and change only the combiner: averaged, they score **0.001554 below**
   the best single model; weighted by a logistic regression, **0.000908 above** it.
   The model I had rejected as the worst blend partner I ever measured takes the
   seventh largest weight of eighteen.
3. **The correction had a second half, and it took me ten more days to see it.**
   Fixing a combiner does not go back and re-examine the conclusions that were reached
   with the broken one. Three rejected models came back when I finally looked, and one
   of them, XGBoost, **had never been run at all**. It is the best single model in the
   competition for me.
4. **A rule for when a reopening is worth the compute, with six points on it and two
   predictions made in advance.** A change of representation revalues *learners*, not
   their *knobs*. Three learner reopenings paid; three hyperparameter sweeps on the
   same representation returned nothing, and the last two were called null in advance.
5. **Fold standard deviation is the wrong bar for a paired comparison** and it almost
   buried my one real improvement.
6. **The public leaderboard here has a standard error near 0.001**, larger than every
   gain after the fourth experiment. Two of my submissions demonstrate this cleanly.

Also, because it is the honest counterweight to point 3: the largest single-model gain
in the whole competition, **+0.026**, was worth **0.000043** to the thing I actually
submit. Effect size and incremental value are different quantities and this notebook
has the cleanest example of that I have produced.

The full experiment ledger, every notebook, and the working log are in the repo
linked at the bottom.


## Setup

The measured results below are inlined as data. They come from the out-of-fold
prediction vectors saved by the training runs in the repo, not from numbers typed
in by hand. The script that produced this blob recomputes every figure from those
vectors, so a number in a chart and a number in the ledger cannot drift apart.

One of the LightGBM rows is retrained live further down, so you can watch a
ledger row reproduce instead of taking the table on trust.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Measured from the saved out-of-fold vectors. See the repo link at the bottom;
# `writeup_numbers.py` regenerates this blob from `artifacts/oof/`.
D = json.loads(r'''{"missing_base_rate":0.7094243168830872,"missingness":[{"feature":"gender","n_missing":29034,"miss_frac":0.04199494047317713,"rate":0.7087897062301636,"rate_present":0.7094521522521973,"lift":-0.0006624460220336914,"z":-0.24333532780246742},{"feature":"stress_level","n_missing":55148,"miss_frac":0.07976637656591487,"rate":0.7090012431144714,"rate_present":0.7094610333442688,"lift":-0.0004597902297973633,"z":-0.22813451817273367},{"feature":"social_media_hours","n_missing":133995,"miss_frac":0.19381111967704656,"rate":0.709414541721344,"rate_present":0.7094267010688782,"lift":-1.2159347534179688e-05,"z":-0.008802181122276542},{"feature":"academic_work_impact","n_missing":44224,"miss_frac":0.06396584168512039,"rate":0.7094563841819763,"rate_present":0.7094221711158752,"lift":3.421306610107422e-05,"z":0.015331483743799402},{"feature":"notifications_per_day","n_missing":67584,"miss_frac":0.0977538767286355,"rate":0.7102420926094055,"rate_present":0.7093357443809509,"lift":0.0009063482284545898,"z":0.4929431994854286},{"feature":"work_study_hours","n_missing":51518,"miss_frac":0.07451592420256042,"rate":0.7106448411941528,"rate_present":0.7093260884284973,"lift":0.0013187527656555176,"z":0.634226942111851},{"feature":"gaming_hours","n_missing":126821,"miss_frac":0.18343460583277527,"rate":0.7105053663253784,"rate_present":0.7091814875602722,"lift":0.0013238787651062012,"z":0.9383321982774018},{"feature":"weekend_screen_time","n_missing":112063,"miss_frac":0.16208855184423948,"rate":0.7106538414955139,"rate_present":0.7091864943504333,"lift":0.0014673471450805664,"z":0.9903310293760929},{"feature":"daily_screen_time_hours","n_missing":95854,"miss_frac":0.13864376331597164,"rate":0.7113004922866821,"rate_present":0.709122359752655,"lift":0.0021781325340270996,"z":1.3784726525679663},{"feature":"app_opens_per_day","n_missing":80710,"miss_frac":0.1167393967620764,"rate":0.7127864956855774,"rate_present":0.7089799642562866,"lift":0.0038065314292907715,"z":2.2384884320856524},{"feature":"age","n_missing":28929,"miss_frac":0.04184306788415448,"rate":0.7134017944335938,"rate_present":0.7092506289482117,"lift":0.00415116548538208,"z":1.5222024319584473},{"feature":"sleep_hours","n_missing":44480,"miss_frac":0.06433612152121371,"rate":0.7133768200874329,"rate_present":0.7091525793075562,"lift":0.004224240779876709,"z":1.8980529200639886}],"single_cv":{"anchor (100 trees)":{"mean":0.9549467010256301,"sd":0.00064490396666483},"300 trees":{"mean":0.9606048423752365,"sd":0.0006875902743794306},"1000 trees":{"mean":0.9621409051321341,"sd":0.0008590300603214384},"2000 trees":{"mean":0.9618323239022759,"sd":0.000952472720968497},"lr 0.10":{"mean":0.9621982367135239,"sd":0.0008159296161059919},"lr 0.05":{"mean":0.9632098272187225,"sd":0.000591461409111401},"lr 0.03":{"mean":0.9632745391279285,"sd":0.0005493987703548775},"bagged seed 42":{"mean":0.9634705327026813,"sd":0.0005907357588788866},"bagged seed 2024":{"mean":0.9632336420315293,"sd":0.0008994906856684058},"bagged seed 7":{"mean":0.963445428051234,"sd":0.0004776115887212625},"bagged seed 2025":{"mean":0.9633374484598652,"sd":0.0007310625216525802},"bagged seed 13":{"mean":0.9634827033576115,"sd":0.0005519140335981485}},"pairs":[{"a":"anchor (100 trees)","b":"300 trees","spearman":0.9930765017385362,"cv_a":0.9549467010256301,"cv_b":0.9606048423752365,"cv_gap":0.005658141349606405,"blend":0.9584202404520668,"gain":-0.0021846019231697156,"paired_sd":0.00014357766590540974,"folds_won":0},{"a":"anchor (100 trees)","b":"1000 trees","spearman":0.9836829150135086,"cv_a":0.9549467010256301,"cv_b":0.9621409051321341,"cv_gap":0.007194204106504065,"blend":0.9600452697116528,"gain":-0.002095635420481301,"paired_sd":0.0002671712392164481,"folds_won":0},{"a":"anchor (100 trees)","b":"2000 trees","spearman":0.9745648778947578,"cv_a":0.9549467010256301,"cv_b":0.9618323239022759,"cv_gap":0.006885622876645847,"blend":0.9602860398817803,"gain":-0.0015462840204955696,"paired_sd":0.00014914676593473817,"folds_won":0},{"a":"anchor (100 trees)","b":"lr 0.10","spearman":0.984439257835441,"cv_a":0.9549467010256301,"cv_b":0.9621982367135239,"cv_gap":0.007251535687893829,"blend":0.960145659778119,"gain":-0.0020525769354049483,"paired_sd":0.00027167713997864815,"folds_won":0},{"a":"anchor (100 trees)","b":"lr 0.05","spearman":0.9890902353771174,"cv_a":0.9549467010256301,"cv_b":0.9632098272187225,"cv_gap":0.00826312619309244,"blend":0.96043968609283,"gain":-0.002770141125892489,"paired_sd":8.846259614537647e-05,"folds_won":0},{"a":"anchor (100 trees)","b":"lr 0.03","spearman":0.9888691181011995,"cv_a":0.9549467010256301,"cv_b":0.9632745391279285,"cv_gap":0.008327838102298424,"blend":0.9604384451434804,"gain":-0.0028360939844481294,"paired_sd":7.756749282176583e-05,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 42","spearman":0.9883336427547597,"cv_a":0.9549467010256301,"cv_b":0.9634705327026813,"cv_gap":0.008523831677051286,"blend":0.9606445874293545,"gain":-0.0028259452733267352,"paired_sd":6.278914037092448e-05,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 2024","spearman":0.9834789992708939,"cv_a":0.9549467010256301,"cv_b":0.9632336420315293,"cv_gap":0.008286941005899218,"blend":0.9606286456529392,"gain":-0.0026049963785901966,"paired_sd":0.0004952931952778384,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 7","spearman":0.9870274795339734,"cv_a":0.9549467010256301,"cv_b":0.963445428051234,"cv_gap":0.008498727025603947,"blend":0.9607032757016605,"gain":-0.0027421523495735345,"paired_sd":0.00019378306930941934,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 2025","spearman":0.9856464416771401,"cv_a":0.9549467010256301,"cv_b":0.9633374484598652,"cv_gap":0.00839074743423518,"blend":0.9606608259490772,"gain":-0.0026766225107882403,"paired_sd":0.0003754351886575923,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 13","spearman":0.9876338741521341,"cv_a":0.9549467010256301,"cv_b":0.9634827033576115,"cv_gap":0.008536002331981485,"blend":0.9606766054730324,"gain":-0.0028060978845790173,"paired_sd":8.744022263481383e-05,"folds_won":0},{"a":"300 trees","b":"1000 trees","spearman":0.9885054227814063,"cv_a":0.9606048423752365,"cv_b":0.9621409051321341,"cv_gap":0.0015360627568976604,"blend":0.9619497164809061,"gain":-0.00019118865122798035,"paired_sd":0.0003105898197343192,"folds_won":1},{"a":"300 trees","b":"2000 trees","spearman":0.9787488030888335,"cv_a":0.9606048423752365,"cv_b":0.9618323239022759,"cv_gap":0.0012274815270394424,"blend":0.9621155630162603,"gain":0.0002832391139844681,"paired_sd":0.00025752330111137597,"folds_won":5},{"a":"300 trees","b":"lr 0.10","spearman":0.9897116101269284,"cv_a":0.9606048423752365,"cv_b":0.9621982367135239,"cv_gap":0.0015933943382874238,"blend":0.9620302270499028,"gain":-0.00016800966362118253,"paired_sd":0.00032083934708392205,"folds_won":1},{"a":"300 trees","b":"lr 0.05","spearman":0.9928298718287766,"cv_a":0.9606048423752365,"cv_b":0.9632098272187225,"cv_gap":0.0026049848434860357,"blend":0.9623782978541178,"gain":-0.0008315293646048883,"paired_sd":3.146865347652024e-05,"folds_won":0},{"a":"300 trees","b":"lr 0.03","spearman":0.9925160458419798,"cv_a":0.9606048423752365,"cv_b":0.9632745391279285,"cv_gap":0.002669696752692019,"blend":0.9623837563003457,"gain":-0.0008907828275825702,"paired_sd":5.81414587942412e-05,"folds_won":0},{"a":"300 trees","b":"bagged seed 42","spearman":0.9917988890272705,"cv_a":0.9606048423752365,"cv_b":0.9634705327026813,"cv_gap":0.002865690327444881,"blend":0.962573015847077,"gain":-0.0008975168556043967,"paired_sd":3.177181594536487e-05,"folds_won":0},{"a":"300 trees","b":"bagged seed 2024","spearman":0.9875189229544078,"cv_a":0.9606048423752365,"cv_b":0.9632336420315293,"cv_gap":0.002628799656292813,"blend":0.9625809392396649,"gain":-0.0006527027918643569,"paired_sd":0.0005196698959231307,"folds_won":1},{"a":"300 trees","b":"bagged seed 7","spearman":0.9910458358371231,"cv_a":0.9606048423752365,"cv_b":0.963445428051234,"cv_gap":0.0028405856759975423,"blend":0.9626551901841743,"gain":-0.0007902378670594957,"paired_sd":0.00016339046508889087,"folds_won":0},{"a":"300 trees","b":"bagged seed 2025","spearman":0.9896924463922334,"cv_a":0.9606048423752365,"cv_b":0.9633374484598652,"cv_gap":0.0027326060846287747,"blend":0.9626115939392029,"gain":-0.0007258545206625833,"paired_sd":0.00036815509459561774,"folds_won":0},{"a":"300 trees","b":"bagged seed 13","spearman":0.9915065958175602,"cv_a":0.9606048423752365,"cv_b":0.9634827033576115,"cv_gap":0.00287786098237508,"blend":0.9626346921274094,"gain":-0.000848011230202217,"paired_sd":5.133159390636951e-05,"folds_won":0},{"a":"1000 trees","b":"2000 trees","spearman":0.9838366777610164,"cv_a":0.9621409051321341,"cv_b":0.9618323239022759,"cv_gap":0.000308581229858218,"blend":0.9626444131158106,"gain":0.0005035079836765322,"paired_sd":0.0003067286024402627,"folds_won":5},{"a":"1000 trees","b":"lr 0.10","spearman":0.993022042353452,"cv_a":0.9621409051321341,"cv_b":0.9621982367135239,"cv_gap":5.7331581389763464e-05,"blend":0.9625869982602765,"gain":0.00038876154675240306,"paired_sd":0.00037827810147254804,"folds_won":4},{"a":"1000 trees","b":"lr 0.05","spearman":0.988195036539881,"cv_a":0.9621409051321341,"cv_b":0.9632098272187225,"cv_gap":0.0010689220865883753,"blend":0.9631437658770949,"gain":-6.606134162769894e-05,"paired_sd":0.00011184498479289316,"folds_won":1},{"a":"1000 trees","b":"lr 0.03","spearman":0.9872628803978186,"cv_a":0.9621409051321341,"cv_b":0.9632745391279285,"cv_gap":0.0011336339957943586,"blend":0.9631526335033342,"gain":-0.00012190562459422072,"paired_sd":0.00010329971947249546,"folds_won":1},{"a":"1000 trees","b":"bagged seed 42","spearman":0.986384154208615,"cv_a":0.9621409051321341,"cv_b":0.9634705327026813,"cv_gap":0.0013296275705472205,"blend":0.96335338176835,"gain":-0.00011715093433148916,"paired_sd":9.469678664213163e-05,"folds_won":1},{"a":"1000 trees","b":"bagged seed 2024","spearman":0.9857334506794263,"cv_a":0.9621409051321341,"cv_b":0.9632336420315293,"cv_gap":0.0010927368993951525,"blend":0.9633854237034913,"gain":0.00015178167196217007,"paired_sd":0.0005793157968198004,"folds_won":2},{"a":"1000 trees","b":"bagged seed 7","spearman":0.9855514711213312,"cv_a":0.9621409051321341,"cv_b":0.963445428051234,"cv_gap":0.001304522919099882,"blend":0.9634484920717767,"gain":3.0640205427756586e-06,"paired_sd":0.0002697152769427477,"folds_won":2},{"a":"1000 trees","b":"bagged seed 2025","spearman":0.9862711547093388,"cv_a":0.9621409051321341,"cv_b":0.9633374484598652,"cv_gap":0.0011965433277311144,"blend":0.9634056263858917,"gain":6.81779260263582e-05,"paired_sd":0.00039237778156615686,"folds_won":2},{"a":"1000 trees","b":"bagged seed 13","spearman":0.98603856433902,"cv_a":0.9621409051321341,"cv_b":0.9634827033576115,"cv_gap":0.0013417982254774197,"blend":0.963429531490062,"gain":-5.317186754929537e-05,"paired_sd":0.00012923666079599753,"folds_won":1},{"a":"2000 trees","b":"lr 0.10","spearman":0.9833150022204985,"cv_a":0.9618323239022759,"cv_b":0.9621982367135239,"cv_gap":0.0003659128112479815,"blend":0.9626918518433328,"gain":0.0004936151298087887,"paired_sd":0.00031259624859980735,"folds_won":5},{"a":"2000 trees","b":"lr 0.05","spearman":0.979252069284783,"cv_a":0.9618323239022759,"cv_b":0.9632098272187225,"cv_gap":0.0013775033164465933,"blend":0.9631591077331179,"gain":-5.071948560479989e-05,"paired_sd":0.00018701065513062178,"folds_won":2},{"a":"2000 trees","b":"lr 0.03","spearman":0.9784559497000986,"cv_a":0.9618323239022759,"cv_b":0.9632745391279285,"cv_gap":0.0014422152256525766,"blend":0.9631730297189047,"gain":-0.00010150940902369232,"paired_sd":0.00018153760995878206,"folds_won":2},{"a":"2000 trees","b":"bagged seed 42","spearman":0.9778607195685959,"cv_a":0.9618323239022759,"cv_b":0.9634705327026813,"cv_gap":0.0016382088004054385,"blend":0.9633665969708053,"gain":-0.00010393573187614802,"paired_sd":0.0001713721615862099,"folds_won":2},{"a":"2000 trees","b":"bagged seed 2024","spearman":0.9774052957938391,"cv_a":0.9618323239022759,"cv_b":0.9632336420315293,"cv_gap":0.0014013181292533705,"blend":0.9634046403472538,"gain":0.00017099831572446876,"paired_sd":0.0006024839590794951,"folds_won":3},{"a":"2000 trees","b":"bagged seed 7","spearman":0.9772698905921975,"cv_a":0.9618323239022759,"cv_b":0.963445428051234,"cv_gap":0.0016131041489581,"blend":0.9634685934347784,"gain":2.3165383544276884e-05,"paired_sd":0.00032083577357150725,"folds_won":2},{"a":"2000 trees","b":"bagged seed 2025","spearman":0.9773175578834753,"cv_a":0.9618323239022759,"cv_b":0.9633374484598652,"cv_gap":0.0015051245575893324,"blend":0.9634248963271659,"gain":8.744786730061627e-05,"paired_sd":0.0004201743134994768,"folds_won":3},{"a":"2000 trees","b":"bagged seed 13","spearman":0.9776912226772052,"cv_a":0.9618323239022759,"cv_b":0.9634827033576115,"cv_gap":0.0016503794553356377,"blend":0.963449722416399,"gain":-3.298094121246819e-05,"paired_sd":0.00019313218884922797,"folds_won":2},{"a":"lr 0.10","b":"lr 0.05","spearman":0.9892230127072061,"cv_a":0.9621982367135239,"cv_b":0.9632098272187225,"cv_gap":0.0010115905051986118,"blend":0.9631919295609059,"gain":-1.789765781663455e-05,"paired_sd":7.606636672390308e-05,"folds_won":1},{"a":"lr 0.10","b":"lr 0.03","spearman":0.9880457011882899,"cv_a":0.9621982367135239,"cv_b":0.9632745391279285,"cv_gap":0.0010763024144045952,"blend":0.9632010889645475,"gain":-7.345016338087263e-05,"paired_sd":7.389008662888954e-05,"folds_won":1},{"a":"lr 0.10","b":"bagged seed 42","spearman":0.9875360821450077,"cv_a":0.9621982367135239,"cv_b":0.9634705327026813,"cv_gap":0.001272295989157457,"blend":0.9634040903219374,"gain":-6.644238074391407e-05,"paired_sd":7.295835529471971e-05,"folds_won":1},{"a":"lr 0.10","b":"bagged seed 2024","spearman":0.9869777426626942,"cv_a":0.9621982367135239,"cv_b":0.9632336420315293,"cv_gap":0.001035405318005389,"blend":0.963437960593061,"gain":0.00020431856153169115,"paired_sd":0.0005174098791223362,"folds_won":3},{"a":"lr 0.10","b":"bagged seed 7","spearman":0.9865373605534884,"cv_a":0.9621982367135239,"cv_b":0.963445428051234,"cv_gap":0.0012471913377101185,"blend":0.9635004824253564,"gain":5.5054374122454064e-05,"paired_sd":0.00023473099132235637,"folds_won":3},{"a":"lr 0.10","b":"bagged seed 2025","spearman":0.9873094081837185,"cv_a":0.9621982367135239,"cv_b":0.9633374484598652,"cv_gap":0.001139211746341351,"blend":0.9634579694930135,"gain":0.00012052103314810214,"paired_sd":0.0003227881298622006,"folds_won":2},{"a":"lr 0.10","b":"bagged seed 13","spearman":0.9872940895721637,"cv_a":0.9621982367135239,"cv_b":0.9634827033576115,"cv_gap":0.0012844666440876562,"blend":0.9634802727457139,"gain":-2.4306118975081502e-06,"paired_sd":9.735444608424457e-05,"folds_won":2},{"a":"lr 0.05","b":"lr 0.03","spearman":0.998091892385352,"cv_a":0.9632098272187225,"cv_b":0.9632745391279285,"cv_gap":6.471190920598335e-05,"blend":0.9633468399038323,"gain":7.230077590392181e-05,"paired_sd":4.177970762279755e-05,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 42","spearman":0.9957503870292943,"cv_a":0.9632098272187225,"cv_b":0.9634705327026813,"cv_gap":0.0002607054839588452,"blend":0.9635728739572048,"gain":0.00010234125452337483,"paired_sd":2.0210071213462815e-05,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 2024","spearman":0.9913523243456657,"cv_a":0.9632098272187225,"cv_b":0.9632336420315293,"cv_gap":2.381481280677722e-05,"blend":0.963601173661026,"gain":0.00036753162949680894,"paired_sd":0.0005356904441392467,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 7","spearman":0.994866027966549,"cv_a":0.9632098272187225,"cv_b":0.963445428051234,"cv_gap":0.00023560083251150665,"blend":0.963670168938625,"gain":0.00022474088739101727,"paired_sd":0.0001651926687614814,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 2025","spearman":0.9931959904996398,"cv_a":0.9632098272187225,"cv_b":0.9633374484598652,"cv_gap":0.0001276212411427391,"blend":0.9636267335120678,"gain":0.00028928505220238154,"paired_sd":0.00035726285824102155,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 13","spearman":0.9947631273975702,"cv_a":0.9632098272187225,"cv_b":0.9634827033576115,"cv_gap":0.0002728761388890444,"blend":0.9636543963713378,"gain":0.00017169301372617075,"paired_sd":3.172005260813579e-05,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 42","spearman":0.9961549651143419,"cv_a":0.9632745391279285,"cv_b":0.9634705327026813,"cv_gap":0.00019599357475286183,"blend":0.9635807338766404,"gain":0.00011020117395905693,"paired_sd":3.9165531996792825e-05,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 2024","spearman":0.990550712129159,"cv_a":0.9632745391279285,"cv_b":0.9632336420315293,"cv_gap":4.0897096399206134e-05,"blend":0.9636112997552413,"gain":0.00033676062731287094,"paired_sd":0.00014561005900725176,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 7","spearman":0.9947482610126726,"cv_a":0.9632745391279285,"cv_b":0.963445428051234,"cv_gap":0.0001708889233055233,"blend":0.9636787049227694,"gain":0.00023327687153544828,"paired_sd":0.00017757322392750798,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 2025","spearman":0.992860946772584,"cv_a":0.9632745391279285,"cv_b":0.9633374484598652,"cv_gap":6.290933193675574e-05,"blend":0.9636357347513217,"gain":0.0002982862914564066,"paired_sd":0.0003638300855812459,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 13","spearman":0.9948778894230152,"cv_a":0.9632745391279285,"cv_b":0.9634827033576115,"cv_gap":0.00020816422968306103,"blend":0.9636622300383962,"gain":0.00017952668078475843,"paired_sd":3.7339744282254896e-05,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 2024","spearman":0.989781705992632,"cv_a":0.9634705327026813,"cv_b":0.9632336420315293,"cv_gap":0.00023689067115206797,"blend":0.9637088284874322,"gain":0.00023829578475076385,"paired_sd":0.00014183887570779546,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 7","spearman":0.9952482999788735,"cv_a":0.9634705327026813,"cv_b":0.963445428051234,"cv_gap":2.5104651447338533e-05,"blend":0.9637699273723278,"gain":0.0002993946696464134,"paired_sd":7.607100152180388e-05,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 2025","spearman":0.9925200396846862,"cv_a":0.9634705327026813,"cv_b":0.9633374484598652,"cv_gap":0.0001330842428161061,"blend":0.9637342938203286,"gain":0.0002637611176472099,"paired_sd":6.152003333392871e-05,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 13","spearman":0.9963487474204312,"cv_a":0.9634705327026813,"cv_b":0.9634827033576115,"cv_gap":1.2170654930199198e-05,"blend":0.9637588004267232,"gain":0.00027609706911186916,"paired_sd":4.807158876957789e-05,"folds_won":5},{"a":"bagged seed 2024","b":"bagged seed 7","spearman":0.9902793175540823,"cv_a":0.9632336420315293,"cv_b":0.963445428051234,"cv_gap":0.00021178601970472943,"blend":0.9637195903091171,"gain":0.0002741622578831704,"paired_sd":0.00022970901337129355,"folds_won":4},{"a":"bagged seed 2024","b":"bagged seed 2025","spearman":0.992782079445031,"cv_a":0.9632336420315293,"cv_b":0.9633374484598652,"cv_gap":0.00010380642833596188,"blend":0.9636978171807534,"gain":0.00036036872088816006,"paired_sd":0.0002568694265550775,"folds_won":5},{"a":"bagged seed 2024","b":"bagged seed 13","spearman":0.9899107954603206,"cv_a":0.9632336420315293,"cv_b":0.9634827033576115,"cv_gap":0.00024906132608226716,"blend":0.9637077852933901,"gain":0.0002250819357786149,"paired_sd":0.0001353015392851377,"folds_won":4},{"a":"bagged seed 7","b":"bagged seed 2025","spearman":0.9920436551301872,"cv_a":0.963445428051234,"cv_b":0.9633374484598652,"cv_gap":0.00010797959136876756,"blend":0.9637548888019702,"gain":0.0003094607507363234,"paired_sd":0.00016682899948468972,"folds_won":5},{"a":"bagged seed 7","b":"bagged seed 13","spearman":0.9955129368789487,"cv_a":0.963445428051234,"cv_b":0.9634827033576115,"cv_gap":3.727530637753773e-05,"blend":0.9637456075863582,"gain":0.0002629042287467298,"paired_sd":5.067585379873403e-05,"folds_won":5},{"a":"bagged seed 2025","b":"bagged seed 13","spearman":0.9926527702006471,"cv_a":0.9633374484598652,"cv_b":0.9634827033576115,"cv_gap":0.0001452548977463053,"blend":0.9637338506377544,"gain":0.00025114728014290487,"paired_sd":4.913150107965401e-05,"folds_won":5}],"seed_saturation":[{"n_seeds":1,"cv":0.9634705327026813,"sd":0.0005907357588788866},{"n_seeds":2,"cv":0.9637088284874322,"sd":0.0006021893977001631},{"n_seeds":3,"cv":0.9638210471372626,"sd":0.0005601863998334479},{"n_seeds":4,"cv":0.9638575079829785,"sd":0.0005564493204720445},{"n_seeds":5,"cv":0.9638804130611953,"sd":0.0005547070146738952}],"neural":{"cv":0.9391689164198243,"sd":0.000759146589520503,"lgbm_blend_cv":0.9638804130611953,"spearman_vs_blend":0.9650419155042297,"within_family_spearman_min":0.9745648778947578,"within_family_spearman_max":0.998091892385352,"catboost_spearman":0.9877,"weight_curve":[{"w":0.05,"blend":0.9636925088678598,"gain":-0.00018790419333551965,"paired_sd":1.584787372166672e-05,"folds_won":0},{"w":0.1,"blend":0.9634012345881462,"gain":-0.00047917847304914665,"paired_sd":3.266236322796298e-05,"folds_won":0},{"w":0.15,"blend":0.9630017114431585,"gain":-0.0008787016180368257,"paired_sd":5.0449512012949546e-05,"folds_won":0},{"w":0.2,"blend":0.9624893155070259,"gain":-0.0013910975541695514,"paired_sd":6.803900177538933e-05,"folds_won":0},{"w":0.25,"blend":0.9618601560109207,"gain":-0.002020257050274665,"paired_sd":8.61945047688188e-05,"folds_won":0},{"w":0.3,"blend":0.9611110810618296,"gain":-0.0027693319993658204,"paired_sd":0.0001051837996202618,"folds_won":0},{"w":0.4,"blend":0.959244330946718,"gain":-0.004636082114477369,"paired_sd":0.00014656530985861184,"folds_won":0},{"w":0.5,"blend":0.9568859117407996,"gain":-0.006994501320395918,"paired_sd":0.0001917106564771205,"folds_won":0}]},"target_encoding":{"cv":0.9667823810724869,"sd":0.00045339845274297596,"vs_same_model_raw":0.003311848369805359,"paired_sd":0.00027030808391996584,"folds_won":5,"seed_cv":{"te42":0.9667823810724869,"te2024":0.9667707371023573,"te7":0.9667285893376194,"te2025":0.9667433669355111,"te13":0.9667894194900226}},"combiners":[{"name":"best single model","auc":0.9668002879455457,"gain":0.0,"paired_sd":0.0,"splits_won":0},{"name":"rank mean, 5 seeds","auc":0.9672337289639643,"gain":0.0004334410184185566,"paired_sd":1.0266049366342973e-05,"splits_won":5},{"name":"rank mean, all 18","auc":0.9652460727650588,"gain":-0.0015542151804869286,"paired_sd":5.8776663895356795e-05,"splits_won":0},{"name":"logit stack, 5 seeds","auc":0.9672318074586792,"gain":0.0004315195131335825,"paired_sd":9.806374427543902e-06,"splits_won":5},{"name":"logit stack, all 18","auc":0.9677082743259209,"gain":0.0009079863803753918,"paired_sd":1.803318377621806e-05,"splits_won":5},{"name":"logit stack, 17 no neural","auc":0.9676152018717368,"gain":0.0008149139261911298,"paired_sd":1.6729749928232447e-05,"splits_won":5},{"name":"rank mean, best + neural","auc":0.9605283637839381,"gain":-0.006271924161607578,"paired_sd":7.080080688970952e-05,"splits_won":0},{"name":"logit stack, best + neural","auc":0.9668905655408114,"gain":9.027759526580859e-05,"paired_sd":6.80090222539322e-06,"splits_won":5}],"neural_in_stack":{"gain":9.307245418426202e-05,"paired_sd":3.3165068998384373e-06,"splits_won":5},"stack_coefs":[{"name":"te13","cv":0.9667894194900226,"coef":0.1661686536700836},{"name":"te42","cv":0.9667823810724869,"coef":0.1698624269180501},{"name":"te2024","cv":0.9667707371023573,"coef":0.1643876968929748},{"name":"te2025","cv":0.9667433669355111,"coef":0.1550868561135781},{"name":"te7","cv":0.9667285893376194,"coef":0.13707896946315176},{"name":"bagged_seed_13","cv":0.9634827033576115,"coef":0.08513170605223949},{"name":"bagged_seed_42","cv":0.9634705327026813,"coef":0.09490139210354506},{"name":"bagged_seed_7","cv":0.963445428051234,"coef":0.09061994698070062},{"name":"bagged_seed_2025","cv":0.9633374484598652,"coef":0.06822157570398021},{"name":"lr_0.03","cv":0.9632745391279285,"coef":0.1302179659154745},{"name":"bagged_seed_2024","cv":0.9632336420315293,"coef":0.010133192471048148},{"name":"lr_0.05","cv":0.9632098272187225,"coef":0.10965562107933086},{"name":"lr_0.10","cv":0.9621982367135239,"coef":-0.01952483566725401},{"name":"1000_trees","cv":0.9621409051321341,"coef":-0.009580320659257933},{"name":"2000_trees","cv":0.9618323239022759,"coef":-0.011818217381742786},{"name":"300_trees","cv":0.9606048423752365,"coef":-0.13210997492586757},{"name":"anchor_(100_trees)","cv":0.9549467010256301,"coef":-0.36190176503531174},{"name":"neural","cv":0.9391689164198243,"coef":0.11775909426086155}],"reopenings":{"catboost":{"seed_cv":{"42":0.9669148433358663,"2024":0.9669280516202086,"7":0.966920034991914,"2025":0.966915974945189,"13":0.9669220870192101},"mean":0.9669201983824776,"sd":5.286325159849586e-06,"range":1.3208284342258736e-05,"lgb_mean":0.9667628987875995,"lgb_sd":2.5995085418363844e-05,"lgb_range":6.083015240321288e-05,"gap":0.00015729959487809086,"gap_se":1.1863302235075115e-05,"gap_in_se":13.259343120587275,"raw_feature_gap_fold0":-0.001675,"paired_vs_lgb":{"gain":0.00013246226337941812,"sd":7.972073543863323e-05,"folds_won":5}},"neural":{"cv_raw":0.9391689164198243,"cv_te":0.9653730310336984,"paired":{"gain":0.026204114613874042,"sd":0.0005682164942917429,"folds_won":5},"relative":0.0279013861678535,"spearman_vs_catboost":0.9757084977829875,"spearman_vs_lgb_te":0.9826183980908478,"gap_to_best_gbdt":-0.0015550205865101363},"xgboost":{"seed_cv":{"42":0.9670988540690629,"2024":0.9671475188619094,"7":0.9671317090125238,"2025":0.9670992555227189,"13":0.9671106846745335},"mean":0.9671176044281496,"sd":2.138898042233852e-05,"range":4.8664792846486726e-05,"gap":0.00035470564055017917,"gap_se":1.505478627822536e-05,"gap_in_se":23.56098811334248,"gap_vs_catboost":0.0001974060456720883,"paired_vs_lgb":{"gain":0.0003164729965761559,"sd":0.00013359384565875387,"folds_won":5}}},"stack_curve":[{"name":"18: five TE seeds, twelve raw, one neural","n":18,"cv":0.9676498765337225,"sd":0.00043686112054352414,"step":null},{"name":"19: plus CatBoost","n":19,"cv":0.9677503698900916,"sd":0.00043104882124765285,"step":{"gain":0.00010049335636908019,"sd":1.1715994051371814e-05,"folds_won":5}},{"name":"23: plus four CatBoost seeds","n":23,"cv":0.9677640129793517,"sd":0.0004338593766880294,"step":{"gain":1.3643089260306774e-05,"sd":5.256611117706093e-06,"folds_won":5}},{"name":"24: plus the encoded neural model","n":24,"cv":0.9678065664886208,"sd":0.00043185436227538305,"step":{"gain":4.255350926900548e-05,"sd":1.6279736943648698e-05,"folds_won":5}},{"name":"25: plus XGBoost","n":25,"cv":0.9678727925243432,"sd":0.00042378129999345795,"step":{"gain":6.622603572237207e-05,"sd":2.6900975718914918e-05,"folds_won":5}},{"name":"29: plus four XGBoost seeds","n":29,"cv":0.9679251232906712,"sd":0.00042775869938654855,"step":{"gain":5.2330766327934874e-05,"sd":1.2768100666326554e-05,"folds_won":5}}],"stack24_coefs":[{"name":"neural_te","cv":0.9653730310336984,"coef":0.1299882097919767},{"name":"lr_0.03","cv":0.9632745391279285,"coef":0.11854496223830036},{"name":"lr_0.05","cv":0.9632098272187225,"coef":0.09800474301067164},{"name":"neural","cv":0.9391689164198243,"coef":0.09116045580628687},{"name":"bagged_seed_42","cv":0.9634705327026813,"coef":0.08931103690803528},{"name":"bagged_seed_7","cv":0.963445428051234,"coef":0.08471365923667777},{"name":"te13","cv":0.9667894194900226,"coef":0.0843553658848902},{"name":"te2024","cv":0.9667707371023573,"coef":0.08156822000973611},{"name":"bagged_seed_13","cv":0.9634827033576115,"coef":0.08147154040419947},{"name":"te2025","cv":0.9667433669355111,"coef":0.08140711245803105},{"name":"cat2024","cv":0.9669280516202086,"coef":0.07964488965116749},{"name":"te42","cv":0.9667823810724869,"coef":0.07672537489712991},{"name":"cat13","cv":0.9669220870192101,"coef":0.06545977038064613},{"name":"cat2025","cv":0.966915974945189,"coef":0.06517078459094973},{"name":"te7","cv":0.9667285893376194,"coef":0.06196695397700095},{"name":"bagged_seed_2025","cv":0.9633374484598652,"coef":0.05800812741683205},{"name":"cat42","cv":0.9669148433358663,"coef":0.05796469061201069},{"name":"cat7","cv":0.966920034991914,"coef":0.05280507605839593},{"name":"bagged_seed_2024","cv":0.9632336420315293,"coef":0.007698754749644454},{"name":"1000_trees","cv":0.9621409051321341,"coef":0.005048247927275353},{"name":"lr_0.10","cv":0.9621982367135239,"coef":-0.004318318862460611},{"name":"2000_trees","cv":0.9618323239022759,"coef":-0.01096676431629827},{"name":"300_trees","cv":0.9606048423752365,"coef":-0.14436923195634935},{"name":"anchor_(100_trees)","cv":0.9549467010256301,"coef":-0.337831388664877}],"stack29_coefs":[{"name":"neural_te","cv":0.9653730310336984,"coef":0.1350700550356118},{"name":"xgb2024","cv":0.9671475188619094,"coef":0.1193934118548488},{"name":"lr_0.03","cv":0.9632745391279285,"coef":0.10889515484890294},{"name":"xgb7","cv":0.9671317090125238,"coef":0.09903446914477834},{"name":"xgb13","cv":0.9671106846745335,"coef":0.09534553274707358},{"name":"lr_0.05","cv":0.9632098272187225,"coef":0.09053154499723585},{"name":"neural","cv":0.9391689164198243,"coef":0.08107270668384632},{"name":"xgb2025","cv":0.9670992555227189,"coef":0.07835350144293994},{"name":"bagged_seed_7","cv":0.963445428051234,"coef":0.07241386942646076},{"name":"bagged_seed_42","cv":0.9634705327026813,"coef":0.0718591221856653},{"name":"xgb42","cv":0.9670988540690629,"coef":0.07168997804012497},{"name":"bagged_seed_13","cv":0.9634827033576115,"coef":0.07053087956054215},{"name":"cat2024","cv":0.9669280516202086,"coef":0.06379757557745214},{"name":"cat13","cv":0.9669220870192101,"coef":0.05003443303705199},{"name":"cat2025","cv":0.966915974945189,"coef":0.047493013959004564},{"name":"bagged_seed_2025","cv":0.9633374484598652,"coef":0.045998255266402485},{"name":"cat42","cv":0.9669148433358663,"coef":0.04331513568411631},{"name":"cat7","cv":0.966920034991914,"coef":0.03464432141848209},{"name":"te13","cv":0.9667894194900226,"coef":0.008245116593144954},{"name":"bagged_seed_2024","cv":0.9632336420315293,"coef":0.008235893967071407},{"name":"te2025","cv":0.9667433669355111,"coef":0.006703378798812347},{"name":"1000_trees","cv":0.9621409051321341,"coef":0.006424959945818429},{"name":"te2024","cv":0.9667707371023573,"coef":0.004993056800918476},{"name":"te42","cv":0.9667823810724869,"coef":-0.0034834276451835206},{"name":"lr_0.10","cv":0.9621982367135239,"coef":-0.007258577189062805},{"name":"te7","cv":0.9667285893376194,"coef":-0.010448779587656809},{"name":"2000_trees","cv":0.9618323239022759,"coef":-0.010488232563305803},{"name":"300_trees","cv":0.9606048423752365,"coef":-0.11477072428928428},{"name":"anchor_(100_trees)","cv":0.9549467010256301,"coef":-0.3000752473881765}],"ledger":[{"id":1,"utc":"2026-08-04 02:15","name":"lgbm_default_anchor","cv_mean":0.954947,"cv_std":0.000645,"folds":5,"lb_public":0.95594,"lb_private":null,"submitted":"yes","notes":"untuned lgbm defaults, raw features, native cat and nan handling"},{"id":2,"utc":"2026-08-04 05:29","name":"lgbm_trees100","cv_mean":0.954947,"cv_std":0.000645,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"reproducibility re-run of exp 1, same config, not a new idea"},{"id":3,"utc":"2026-08-04 05:29","name":"lgbm_trees300","cv_mean":0.960605,"cv_std":0.000688,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"n_estimators=300, lr default 0.1, no early stopping, all else anchor"},{"id":4,"utc":"2026-08-04 05:29","name":"lgbm_trees1000","cv_mean":0.962141,"cv_std":0.000859,"folds":5,"lb_public":0.96435,"lb_private":null,"submitted":"yes","notes":"n_estimators=1000, lr default 0.1, no early stopping, all else anchor"},{"id":5,"utc":"2026-08-04 05:29","name":"lgbm_trees2000","cv_mean":0.961832,"cv_std":0.000952,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"n_estimators=2000, lr default 0.1, no early stopping, all else anchor"},{"id":6,"utc":"2026-08-04 06:34","name":"lgbm_lr01","cv_mean":0.962198,"cv_std":0.000816,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"lr=0.1, n_estimators=1000 holding lr*n=100, DETERMINISTIC flags on, not comparable at 4dp to rows 1-5"},{"id":7,"utc":"2026-08-04 06:34","name":"lgbm_lr005","cv_mean":0.96321,"cv_std":0.000591,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"lr=0.05, n_estimators=2000 holding lr*n=100, DETERMINISTIC flags on, not comparable at 4dp to rows 1-5"},{"id":8,"utc":"2026-08-04 06:34","name":"lgbm_lr003","cv_mean":0.963275,"cv_std":0.000549,"folds":5,"lb_public":0.96478,"lb_private":null,"submitted":"yes","notes":"lr=0.03, n_estimators=3333 holding lr*n=100, DETERMINISTIC flags on, not comparable at 4dp to rows 1-5"},{"id":9,"utc":"2026-08-04 18:54","name":"lgbm_bag08_seed42","cv_mean":0.963471,"cv_std":0.000591,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"bagged subsample=0.8 freq=1 colsample=0.8, lr=0.05 n=2000, seed 42. One variable against exp7, which is this config with bagging off."},{"id":10,"utc":"2026-08-04 19:01","name":"lgbm_bag08_seed2024","cv_mean":0.963234,"cv_std":0.000899,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"bagged subsample=0.8 freq=1 colsample=0.8, lr=0.05 n=2000, seed 2024. One variable against exp7, which is this config with bagging off."},{"id":11,"utc":"2026-08-04 19:07","name":"lgbm_bag08_seed7","cv_mean":0.963445,"cv_std":0.000478,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"bagged subsample=0.8 freq=1 colsample=0.8, lr=0.05 n=2000, seed 7. One variable against exp7, which is this config with bagging off."},{"id":12,"utc":"2026-08-04 19:07","name":"lgbm_bag08_seedblend3","cv_mean":0.963821,"cv_std":0.00056,"folds":5,"lb_public":0.96509,"lb_private":null,"submitted":"yes","notes":"rank average of bagged seeds [42, 2024, 7]. One variable against the single-seed rows. Capacity-varied floor from 07 is +0.000072."},{"id":13,"utc":"2026-08-04 19:26","name":"lgbm_bag08_seed2025","cv_mean":0.963337,"cv_std":0.000731,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"bagged subsample=0.8 freq=1 colsample=0.8, lr=0.05 n=2000, seed 2025. One variable against exp7, which is this config with bagging off."},{"id":14,"utc":"2026-08-04 19:33","name":"lgbm_bag08_seed13","cv_mean":0.963483,"cv_std":0.000552,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"bagged subsample=0.8 freq=1 colsample=0.8, lr=0.05 n=2000, seed 13. One variable against exp7, which is this config with bagging off."},{"id":15,"utc":"2026-08-04 19:33","name":"lgbm_bag08_seedblend5","cv_mean":0.96388,"cv_std":0.000555,"folds":5,"lb_public":0.96508,"lb_private":null,"submitted":"yes","notes":"rank average of bagged seeds [42, 2024, 7, 2025, 13]. One variable against the single-seed rows. Capacity-varied floor from 07 is +0.000072."},{"id":16,"utc":"2026-08-05 21:07","name":"neural_mlp_kaggle","cv_mean":0.939169,"cv_std":0.000759,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"embedding MLP 512/256/128, quantile transform + missing mask, AdamW OneCycle, early stop on fold AUC. Kaggle T4, 14.9 min, fold alignment verified. Not submitted: 0.0247 behind the lgbm blend and the blend gate said stop at every weight from 0.05 to 0.50, 0/5 folds. Closes the last diversity hypothesis."},{"id":17,"utc":"2026-08-11 23:38","name":"lgbm_bag08_seed42_te","cv_mean":0.966782,"cv_std":0.000453,"folds":5,"lb_public":0.96825,"lb_private":null,"submitted":"yes","notes":"target + frequency encoding on all 12 columns, nested out-of-fold (inner KFold-5, smoothing 10), 12 raw features -> 36. One variable against exp 9: identical model, folds and seed, only the feature set changes. Kaggle CPU 8.6 min, fold alignment verified. Leak checks pass on full data: validation rows 0.0, training rows 8.1e-05 against a prior shift of 1.3e-04, sanity 0.9967. Paired vs exp9 +0.003312, 5/5 folds, sd 0.000270; vs exp15 +0.002902, 5/5 folds. Largest gain since exp 4. Idea from the public notebook by tomasa2, our own implementation and CV."},{"id":18,"utc":"2026-08-11 23:55","name":"lgbm_bag08_seed42_te_imp","cv_mean":0.966805,"cv_std":0.000469,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"median-imputed copies of the 9 numeric columns added ALONGSIDE the originals, medians fit inside the fold loop. One variable against row 17. NULL RESULT: +0.000023, paired sd 0.000032, so 0.7 paired sd, wins 4/5 but the magnitude is nothing. Target encoding was 12 paired sd for comparison. The public notebook that reported +0.0012 for this measured it without full target encoding in the baseline; once every column is target-encoded the NaN level already gets its own encoded value, so the imputed columns look redundant. Hypothesis, not tested. Not submitted, not carried forward."},{"id":19,"utc":"2026-08-12 00:09","name":"lgbm_bag08_seed42_te_lat","cv_mean":0.966651,"cv_std":0.000515,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"decimal lattice: fractional part and first decimal digit of the 6 float columns, digits also target-encoded. One variable against row 17. NEGATIVE: -0.000132, paired sd 0.000112, wins 1/5. The raw signal is real and large (verified here: 8.90 pts of target-rate swing across the first digit of daily_screen_time_hours, 11.54 for weekend_screen_time, all six columns well outside noise), so this is not a case of a phantom effect. It is subsumed. The digit is a deterministic function of the value, so target-encoding the exact value already carries it; the digit column only adds POOLING across integer parts, which pays only when the per-value estimate is noisy, and at ~500 rows per level it is not. Cost 24 extra features and 3 extra minutes for nothing. Not submitted, not carried forward."},{"id":20,"utc":"2026-08-12 00:53","name":"lgbm_bag08_seed2024_te","cv_mean":0.966771,"cv_std":0.000446,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 17's configuration at a different model seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Not submitted on its own; exists to be blended. The five TE seeds span 0.966729 to 0.966789, a range of 6e-05, against the raw-feature seeds' 2.5e-04."},{"id":21,"utc":"2026-08-12 00:53","name":"lgbm_bag08_seed7_te","cv_mean":0.966729,"cv_std":0.000433,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 17's configuration at a different model seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Not submitted on its own; exists to be blended. The five TE seeds span 0.966729 to 0.966789, a range of 6e-05, against the raw-feature seeds' 2.5e-04."},{"id":22,"utc":"2026-08-12 00:53","name":"lgbm_bag08_seed2025_te","cv_mean":0.966743,"cv_std":0.000427,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 17's configuration at a different model seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Not submitted on its own; exists to be blended. The five TE seeds span 0.966729 to 0.966789, a range of 6e-05, against the raw-feature seeds' 2.5e-04."},{"id":23,"utc":"2026-08-12 00:53","name":"lgbm_bag08_seed13_te","cv_mean":0.966789,"cv_std":0.000427,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 17's configuration at a different model seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Not submitted on its own; exists to be blended. The five TE seeds span 0.966729 to 0.966789, a range of 6e-05, against the raw-feature seeds' 2.5e-04."},{"id":24,"utc":"2026-08-12 00:53","name":"stack_logit_18","cv_mean":0.967665,"cv_std":0.000432,"folds":5,"lb_public":0.96897,"lb_private":null,"submitted":"yes","notes":"logistic regression on clipped logits of 18 OOF vectors: the 5 target-encoded seeds, the 12 raw-feature LightGBMs and the neural model. Honest gain measured by fitting on half the OOF rows and scoring on the other half over 5 splits: +0.000908 over te42 alone, paired sd 0.000018, 5/5 folds. The CV recorded here is the stacker's in-sample OOF number and is the optimistic one; the split-half figure is the claim. The 5 TE seeds alone are worth +0.000433, so the 12 weaker members roughly double it. THIS OVERTURNS THE REPO'S DIVERSITY CONCLUSION: anchor takes -0.3619 and trees300 -0.1321 as corrections, and the neural model, rejected in row 16, takes +0.1178. Every earlier combiner was equal-weight and structurally could not use a member as a correction."},{"id":25,"utc":"2026-08-12 03:13","name":"stack_logit_18_oof","cv_mean":0.96765,"cv_std":0.000437,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 24's combiner refit inside the fold loop: for each outer fold the logistic regression is fit on the other four folds of the OOF matrix and scored on the held-out one, so the CV here is out-of-fold for the combiner as well as for the members and IS comparable to every other row in this file, which row 24's was not. One variable against row 24: same 18 members, same folds, same C, only the fitting protocol changes. Paired vs row 17 +0.000867, sd 0.000051, 5/5 folds, against the split-half estimate of +0.000908 recorded in row 24, so the two protocols agree. Optimism removed from row 24's number: only 1.5e-05, which is smaller than expected and is what 18 parameters on 691,369 rows should look like. Coefficients are stable across the five fits, largest fold-to-fold sd 0.0387 (bag2024, the redundant member); the neural model's is 0.1178 +/- 0.0020. NOT SUBMITTED, deliberately: the fold-averaged test ordering correlates with the submitted row 24 at Spearman 0.9999995, so a submission would spend a slot to re-measure an ordering the leaderboard cannot distinguish. Run locally by nbconvert --execute on a clean kernel, notebooks/16_oof_stack.ipynb, outputs kept in the notebook as the run record since there is no Kaggle log for it."},{"id":26,"utc":"2026-08-17 18:01","name":"catboost_te","cv_mean":0.966915,"cv_std":0.000435,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"CatBoost on row 17's exact feature set. One variable against row 17: same nested target+frequency encoder, same folds, same seed 42, same lr*iters=100 budget convention; only the learner changes, LightGBM to CatBoost. The encoder is a copy, so its four functions were fingerprinted through ast.unparse against 13_target_encoding.ipynb and are IDENTICAL (0642e41750ef8bab); that is the config hash this layout otherwise lacks. Kaggle CPU 80 min, fold alignment verified, leak checks reproduce 13's numbers exactly (validation rows 0.0, training rows 8.136e-05 against a prior shift of 1.302e-04). Paired vs row 17 +0.000132, sd 0.000080, 5/5 folds, t(4)=3.72. REPORTED AS PARITY, NOT AS AN IMPROVEMENT: the gain is 0.3 of the fold spread and only one CatBoost seed was run, so it fails two of the three tests in CLAUDE.md, and the narrowest fold is won by 1.4e-05. What is not marginal is the reversal. 06 measured CatBoost 0.001675 BEHIND LightGBM on the raw 12 features, so the representation moved the gap by about 0.0018, ten times the residual difference. Determinism: thread_count=6 gave 0.000e+00 between two in-kernel runs, and fold 0 agreed to the 6 printed decimals between this machine and Kaggle, where LightGBM moves ~3e-5 with thread count. Not submitted on its own; it exists to be stacked and the gate is row 27."},{"id":27,"utc":"2026-08-17 19:05","name":"stack_logit_19_oof","cv_mean":0.96775,"cv_std":0.000431,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"Row 25's stack with CatBoost added as a nineteenth member. One variable against row 25: same logistic combiner, same C, same fold-wise fitting protocol, same folds; only the member set changes, 18 to 19. Row 25 is REFIT in the same run rather than quoted, so the comparison is paired inside one notebook, and the refit reproduces row 25's recorded CV to -1.23e-07. Paired vs row 25 +0.000100, sd 0.000012, 5/5 folds, t(4)=19.18. Real but small: it is a ninth of what the 18-member stack was worth over row 17 (+0.000867), and 0.23 of the fold spread. Also +0.000968 over row 17 alone and +0.000836 over row 26 alone. THE COEFFICIENT IS THE FINDING: catboost_te takes +0.3512, more than double any other member, while the five target-encoded LightGBM seeds fall from about +0.16 each in row 25 to about +0.10 here, so the combiner moved weight onto it rather than adding it on top. The neural model holds at +0.1169 against +0.1178 in row 25, unchanged, which says CatBoost supplies something different again rather than displacing it. Largest fold-to-fold coefficient sd 0.0357, on bag2024, the same redundant member row 25 identified. Spearman of the test ordering against the submitted row 24 is 0.9994618, where row 25 sat at 0.9999995 and was held back for that reason, so unlike row 25 this is a materially different ordering. Run locally by nbconvert --execute on a clean kernel, notebooks/18_stack_19.ipynb, outputs kept in the notebook as the run record."},{"id":28,"utc":"2026-08-18 07:08","name":"catboost_te_seed2024","cv_mean":0.966928,"cv_std":0.000468,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 26's configuration at a different CatBoost seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Encoder fingerprinted against 13 and identical (0642e41750ef8bab), fold alignment verified, leak checks reproduce 13's numbers exactly. Kaggle CPU, 65 min each, one kernel for all four. Not submitted on its own; exists to size the seed spread and to be stacked. THE POINT OF THE SWEEP: the five CatBoost seeds span 1.32e-05 (sd 5.29e-06) against the five LightGBM target-encoded seeds' 6.00e-05 (sd 2.58e-05), so CatBoost is about 5x more stable under its own stochasticity here. Family means 0.966920 against 0.966763, a gap of +0.000157 at 13.4 standard errors, which RESOLVES ROW 26'S PARITY CLAIM UPWARD: one seed could not have produced +0.000132 by chance."},{"id":29,"utc":"2026-08-18 08:13","name":"catboost_te_seed7","cv_mean":0.96692,"cv_std":0.000424,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 26's configuration at a different CatBoost seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Encoder fingerprinted against 13 and identical (0642e41750ef8bab), fold alignment verified, leak checks reproduce 13's numbers exactly. Kaggle CPU, 65 min each, one kernel for all four. Not submitted on its own; exists to size the seed spread and to be stacked. THE POINT OF THE SWEEP: the five CatBoost seeds span 1.32e-05 (sd 5.29e-06) against the five LightGBM target-encoded seeds' 6.00e-05 (sd 2.58e-05), so CatBoost is about 5x more stable under its own stochasticity here. Family means 0.966920 against 0.966763, a gap of +0.000157 at 13.4 standard errors, which RESOLVES ROW 26'S PARITY CLAIM UPWARD: one seed could not have produced +0.000132 by chance."},{"id":30,"utc":"2026-08-18 09:18","name":"catboost_te_seed2025","cv_mean":0.966916,"cv_std":0.000431,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 26's configuration at a different CatBoost seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Encoder fingerprinted against 13 and identical (0642e41750ef8bab), fold alignment verified, leak checks reproduce 13's numbers exactly. Kaggle CPU, 65 min each, one kernel for all four. Not submitted on its own; exists to size the seed spread and to be stacked. THE POINT OF THE SWEEP: the five CatBoost seeds span 1.32e-05 (sd 5.29e-06) against the five LightGBM target-encoded seeds' 6.00e-05 (sd 2.58e-05), so CatBoost is about 5x more stable under its own stochasticity here. Family means 0.966920 against 0.966763, a gap of +0.000157 at 13.4 standard errors, which RESOLVES ROW 26'S PARITY CLAIM UPWARD: one seed could not have produced +0.000132 by chance."},{"id":31,"utc":"2026-08-18 10:23","name":"catboost_te_seed13","cv_mean":0.966922,"cv_std":0.000442,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 26's configuration at a different CatBoost seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Encoder fingerprinted against 13 and identical (0642e41750ef8bab), fold alignment verified, leak checks reproduce 13's numbers exactly. Kaggle CPU, 65 min each, one kernel for all four. Not submitted on its own; exists to size the seed spread and to be stacked. THE POINT OF THE SWEEP: the five CatBoost seeds span 1.32e-05 (sd 5.29e-06) against the five LightGBM target-encoded seeds' 6.00e-05 (sd 2.58e-05), so CatBoost is about 5x more stable under its own stochasticity here. Family means 0.966920 against 0.966763, a gap of +0.000157 at 13.4 standard errors, which RESOLVES ROW 26'S PARITY CLAIM UPWARD: one seed could not have produced +0.000132 by chance."},{"id":32,"utc":"2026-08-18 08:40","name":"stack_logit_23_oof","cv_mean":0.967764,"cv_std":0.000434,"folds":5,"lb_public":0.96902,"lb_private":null,"submitted":"yes","notes":"Row 27's stack with the four extra CatBoost seeds added. One variable against row 27: same combiner, same C, same fold-wise protocol, same folds; only the member set changes, 19 to 23. Row 27 is REFIT in the same run rather than quoted and reproduces its recorded CV to +3.70e-07. Paired vs row 27 +0.000014, sd 0.000005, 5/5 folds, t(4)=5.80. CERTAIN AND NEGLIGIBLE, and the ledger should be read that way: it is smaller than the fifth LightGBM seed on the raw features (+0.000022), which this repo already called the point where more seeds becomes activity rather than progress. The t statistic is large only because the stacker is deterministic given its inputs, so the paired spread measures fold-to-fold variation in a fixed quantity rather than run-to-run noise; it says the gain is real, not that it matters. MECHANISM, as predicted in the notebook header before running: the combiner SPLIT row 27's single CatBoost weight rather than adding to it. cat42 falls from +0.3512 to +0.0753 and the five CatBoost coefficients sum to +0.4016, only +0.0504 above row 27's single member, which is what a seed spread of 1.32e-05 should produce. Spearman of the test ordering against row 27 is 0.9999832, so the choice between row 27 and row 32 is immaterial to the leaderboard and neither is worth a slot over the other. CatBoost seed averaging saturates immediately and is closed. Run locally by nbconvert --execute on a clean kernel, notebooks/20_stack_23.ipynb, outputs kept in the notebook as the run record. LB 0.96902, best in the repo, against row 24's 0.96897. CV moved +0.000099 and LB moved +0.000050, same direction, so CV and LB now agree on 8 of 8 submissions. The LB step is well inside the ~0.001 public standard error and was predicted to be, so it confirms the ordering rather than discriminating: it is a tracking data point, not evidence that row 32 beats row 24 on the private split. CV-to-LB offset +0.001256 against row 24's +0.001305."},{"id":33,"utc":"2026-08-18 16:12","name":"neural_te","cv_mean":0.965373,"cv_std":0.000405,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"The row 16 embedding MLP on the target-encoded features. One variable against row 16: identical architecture, hyperparameters, folds, seed and early-stopping rule (512/256/128, BatchNorm, SiLU, 0.2 dropout, 8-dim embeddings, AdamW under OneCycle, 30 epochs, patience 5, batch 4096); only the feature set changes, the numeric block going from 9 raw + 9 mask to 33 (9 raw + 24 encoded) + 9 mask. Encoder fingerprinted against 13 and identical (0642e41750ef8bab), fit inside the fold loop, with the quantile transform and median imputation also fit on training rows only inside the same loop. Kaggle T4, 16.1 min, fold alignment verified on all five checks. Paired vs row 16 +0.026204, sd 0.000568, 5/5 folds, t=103. THE >2% RULE IN CLAUDE.md FIRED AT +2.7901% AND WAS INVESTIGATED RATHER THAN CELEBRATED. Not a leak, and the argument is that the new number is BELOW both GBDTs on the identical encoder (row 17 LightGBM 0.966782, row 26 CatBoost 0.966915), while 13's deliberate no-nesting sanity check scores 0.9967, so a leaking encoder is loud here and three LB-confirmed models already use this exact one. Leak checks reproduce 13's numbers exactly (validation 0.0, training 8.130e-05 against a prior shift of 1.302e-04) and row 16's own vector reproduces its ledger CV to -8.36e-08, so the baseline is right too. The rule fired because the DENOMINATOR was weak, not because the numerator is anomalous. The neural model goes from 0.0247 behind the best GBDT to 0.0015 behind. Spearman 0.9826 vs row 17 and 0.9757 vs row 26, the latter near the bottom of the 0.9741 to 0.9981 band LightGBM produces against itself, so it is still differently wrong while now being strong. Not submitted on its own; the decision is what a fitted combiner does with it, which is row 34. The notebook header predicted an MLP would gain little from a representation that only removes a many-splits cost trees were paying. That prediction was recorded before the run and was wrong by a wide margin."},{"id":34,"utc":"2026-08-18 09:52","name":"stack_logit_24_oof","cv_mean":0.967807,"cv_std":0.000432,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"Row 32's stack with neural_te added as a twenty-fourth member. One variable against row 32: same combiner, same C, same fold-wise protocol, same folds; only the member set changes, 23 to 24. The old raw-feature neural model is KEPT rather than replaced, since removing it would make this two variables. Row 32 is refit in the same run and reproduces its recorded CV to +1.30e-08. Paired vs row 32 +0.000043, sd 0.000016, 5/5 folds, t(4)=5.84. Real, and small: three times the last CatBoost-seed addition (+0.000014) but under half the original CatBoost addition (+0.000100), against a single-model improvement of +0.026204. THE COEFFICIENTS ARE THE RESULT, NOT THE CV. neural_te takes +0.1305, the largest positive weight in the whole 24-member stack, above every GBDT in it. The old neural model does NOT collapse: it goes from +0.1181 to +0.0905, keeping 77 percent of its weight, so the two neural models are wrong in different directions FROM EACH OTHER as well as from the trees, rather than being one mechanism at two strengths. The weight neural_te takes comes mostly out of the GBDTs, with the five CatBoost coefficients falling from +0.4016 to +0.3212 and the TE seeds dropping in step. WHAT THIS SAYS ABOUT STACKING: a 0.026 single-model gain bought 0.000043 in the stack, because the weak neural model at +0.118 was already supplying most of the neural direction and the combiner only needed a cleaner version of a signal it already had. Spearman of the test ordering against the submitted row 32 is 0.9998335, so the leaderboard cannot resolve this and it is not worth a slot over row 32; it is however the CV-best row in the file and therefore a final-selection candidate. Run locally by nbconvert --execute on a clean kernel, notebooks/22_stack_24.ipynb, outputs kept in the notebook as the run record."},{"id":35,"utc":"2026-08-19 20:59","name":"te_leaves15","cv_mean":0.966016,"cv_std":0.000799,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"num_leaves=15, against LightGBM's default of 31. One variable against row 17: the same nested target and frequency encoder (fingerprint 0642e41750ef8bab, verified identical to 13), the same five folds, the same seed 42, the same lr=0.05 and n_estimators=2000 and the same bagging fractions; only num_leaves changes. All four arms trained on the SAME encoded matrices, the encoder being built once per fold, so the arms cannot differ through the encoder's inner split either. Kaggle CPU, 24m 25s for the whole sweep. Leak checks PASS, fold alignment verified, determinism OK. Not submitted, not carried forward. NEGATIVE: paired against the num_leaves=31 arm from the same kernel -0.000766, paired sd 0.000703, 0/5 folds. It is also the only arm whose fold spread degrades, 0.000799 against 31's 0.000453, so it is less stable as well as worse. Underfitting at a fixed tree budget. THE POINT OF THE SWEEP: num_leaves had never appeared in any notebook in this repo, so it was the largest never-searched knob here, and NOTES.md closed tuning on 2026-08-04 under the raw 12 features, before the representation that mattered existed. Re-run under the target-encoded feature set the answer is that LightGBM's default 31 was already at the optimum: the curve is flat from 31 to 63 and falls on both sides. The reopening reasoning that paid for CatBoost (row 26) and the neural model (row 33) does not pay a third time, and num_leaves is closed. PROVENANCE: the num_leaves=31 arm reproduces ledger row 17 BIT-IDENTICALLY, +0.00e+00 on every one of the five folds and +3.81e-07 on the mean, which is the tightest reproduction in this file and confirms deterministic=True holds across Kaggle sessions eight days apart at a matched thread count."},{"id":36,"utc":"2026-08-19 20:59","name":"te_leaves63","cv_mean":0.966758,"cv_std":0.00044,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"num_leaves=63, against LightGBM's default of 31. One variable against row 17: the same nested target and frequency encoder (fingerprint 0642e41750ef8bab, verified identical to 13), the same five folds, the same seed 42, the same lr=0.05 and n_estimators=2000 and the same bagging fractions; only num_leaves changes. All four arms trained on the SAME encoded matrices, the encoder being built once per fold, so the arms cannot differ through the encoder's inner split either. Kaggle CPU, 24m 25s for the whole sweep. Leak checks PASS, fold alignment verified, determinism OK. Not submitted, not carried forward. NULL: paired against the num_leaves=31 arm -0.000024, paired sd 0.000077, 3/5 folds, so it is indistinguishable from the default in both sign and magnitude, and 3/5 is a coin flip. Fold spread 0.000440 against 31's 0.000453 is indistinguishable too. This is the arm that makes the result a plateau rather than a peak: doubling the leaves costs nothing and buys nothing. THE POINT OF THE SWEEP: num_leaves had never appeared in any notebook in this repo, so it was the largest never-searched knob here, and NOTES.md closed tuning on 2026-08-04 under the raw 12 features, before the representation that mattered existed. Re-run under the target-encoded feature set the answer is that LightGBM's default 31 was already at the optimum: the curve is flat from 31 to 63 and falls on both sides. The reopening reasoning that paid for CatBoost (row 26) and the neural model (row 33) does not pay a third time, and num_leaves is closed. PROVENANCE: the num_leaves=31 arm reproduces ledger row 17 BIT-IDENTICALLY, +0.00e+00 on every one of the five folds and +3.81e-07 on the mean, which is the tightest reproduction in this file and confirms deterministic=True holds across Kaggle sessions eight days apart at a matched thread count."},{"id":37,"utc":"2026-08-19 20:59","name":"te_leaves127","cv_mean":0.966596,"cv_std":0.000454,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"num_leaves=127, against LightGBM's default of 31. One variable against row 17: the same nested target and frequency encoder (fingerprint 0642e41750ef8bab, verified identical to 13), the same five folds, the same seed 42, the same lr=0.05 and n_estimators=2000 and the same bagging fractions; only num_leaves changes. All four arms trained on the SAME encoded matrices, the encoder being built once per fold, so the arms cannot differ through the encoder's inner split either. Kaggle CPU, 24m 25s for the whole sweep. Leak checks PASS, fold alignment verified, determinism OK. Not submitted, not carried forward. NEGATIVE AND RELIABLE: paired against the num_leaves=31 arm -0.000186, paired sd 0.000063, 0/5 folds, mean/sd -2.9. It loses every fold by a consistent margin, which is a cleaner signal than its size suggests: the per-fold differences run -0.000217 to -0.000121 with no crossing. That is what too much capacity at a fixed tree count looks like. THE POINT OF THE SWEEP: num_leaves had never appeared in any notebook in this repo, so it was the largest never-searched knob here, and NOTES.md closed tuning on 2026-08-04 under the raw 12 features, before the representation that mattered existed. Re-run under the target-encoded feature set the answer is that LightGBM's default 31 was already at the optimum: the curve is flat from 31 to 63 and falls on both sides. The reopening reasoning that paid for CatBoost (row 26) and the neural model (row 33) does not pay a third time, and num_leaves is closed. PROVENANCE: the num_leaves=31 arm reproduces ledger row 17 BIT-IDENTICALLY, +0.00e+00 on every one of the five folds and +3.81e-07 on the mean, which is the tightest reproduction in this file and confirms deterministic=True holds across Kaggle sessions eight days apart at a matched thread count."},{"id":38,"utc":"2026-08-19 21:28","name":"xgb_te","cv_mean":0.967099,"cv_std":0.000414,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"XGBoost on row 17's exact feature set. One variable against row 17: the same nested target and frequency encoder, the same folds, the same seed 42 and the same lr*n_estimators=100 budget convention that 06 and 17 used; only the learner changes. XGBoost keeps its own tree-structure defaults (hist, max_depth=6, enable_categorical), exactly as 17 kept CatBoost's, so this measures what the learner does out of the box on this representation. The encoder was lifted out of 23 by content match rather than retyped and fingerprints 0642e41750ef8bab, identical to 13. Kaggle CPU, 11m 30s. Leak checks reproduce 13's numbers, fold alignment verified, determinism bit-identical, and row 17's saved vector reproduces its ledger CV to +3.81e-07 in this kernel. THE BEST SINGLE MODEL IN THIS REPO: 0.967099 against CatBoost's 0.966915 (row 26) and LightGBM's 0.966782 (row 17), on the identical encoder and folds. Paired vs row 17 +0.000316, paired sd 0.000134, 5/5 folds, t(4)=5.27, which is 2.4x the gain CatBoost showed over the same baseline. REPORTED AS ONE SEED, NOT AS AN IMPROVEMENT. CLAUDE.md requires a gain to hold across at least two seeds and this is one, which is exactly the ground on which row 26 was logged as parity despite 5/5 folds and t=3.72. Row 26 was later resolved upward by the sweep in rows 28 to 31, and 25_xgb_seeds.ipynb is the same step here. Until it lands this row claims a measurement, not a conclusion. THE PREDICTION IN THE NOTEBOOK HEADER WAS RECORDED BEFORE THE RUN AND WAS WRONG. It argued for a low prior on three named grounds: CatBoost's ordered-target-statistic mechanism does not transfer to XGBoost, the row 34 ceiling sits on top, and num_leaves had just returned a null under this same representation (rows 35 to 37). The header also recorded the counterargument, that both reopenings which did pay had been argued down in advance on reasoning just as specific, and that is the half that held. Not submitted, and the single-model CV is deliberately not the gate: rows 16 and 33 show a model 0.0247 behind still earning +0.1178 in a fitted stack, so the decision is the stack contribution and that is a separate notebook and a separate ledger row because it changes the member set rather than the learner."},{"id":39,"utc":"2026-08-19 21:42","name":"stack_logit_25_oof","cv_mean":0.967873,"cv_std":0.000424,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"Row 34's stack with xgb_te added as a twenty-fifth member. One variable against row 34: same logistic combiner, same C, same fold-wise protocol, same folds; only the member set changes, 24 to 25. Row 34 is REFIT in the same run rather than quoted and reproduces its recorded CV to -4.34e-07. Paired vs row 34 +0.000066, sd 0.000027, 5/5 folds, t(4)=5.50. THE PRE-REGISTERED GATE PASSED ON BOTH CONDITIONS. 24_xgboost_te.ipynb required, before XGBoost had been run, at least 4 of 5 folds positive and at least +0.00005 on the mean, which is the floor 17_catboost_te.ipynb set for CatBoost. The result is 5/5 and +0.000066. It is the second largest stack addition in this repo, above row 34's neural_te (+0.000043) and row 32's four CatBoost seeds (+0.000014), and below row 27's original CatBoost (+0.000100). THE COEFFICIENTS ARE THE RESULT, AND THEY SAY SUBSTITUTION. xgb_te takes +0.2751, more than double the next largest member (neural_te at +0.1316) and the largest single weight ever recorded in this stack. The other 21 GBDT coefficients fall from +1.0900 to +0.7925, a drop of -0.2975 that almost exactly offsets it, so the combiner moved weight onto XGBoost rather than adding it on top. That is row 32's mechanism again in a sharper form. WHAT IT DISPLACES IS SPECIFIC AND INFORMATIVE. The five LightGBM target-encoded seeds lose about -0.045 each, roughly -0.229 of the -0.2975 total, while the five CatBoost seeds lose only about -0.0088 each. XGBoost is displacing the LightGBM seeds in particular, which is what should happen when a strictly better model occupies the same position on the same feature set. The two neural members barely move (neural_te +0.0012, neural -0.0064), confirming again that they supply a direction the trees do not. Largest fold-to-fold coefficient sd 0.0340, on bag2024, the same redundant member rows 25 and 27 identified. CV 0.967873 is the best in this file. NOT SUBMITTED, on the precedent rows 25 and 34 set: Spearman of the test ordering against the submitted row 32 is 0.9998222, against row 34's own 0.9998335, so the public leaderboard cannot resolve the difference and a slot spent here would re-measure an ordering rather than discriminate between two. Run locally by nbconvert --execute on a clean kernel, notebooks/26_stack_25.ipynb, outputs kept in the notebook as the run record. Note that xgb_te itself is still a single seed at this point; 25_xgb_seeds.ipynb is the sweep that resolves that, and it does not change this row, which is about the member set."},{"id":40,"utc":"2026-08-19 22:35","name":"xgb_te_seed2024","cv_mean":0.967148,"cv_std":0.000432,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 38's configuration at a different XGBoost seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Encoder fingerprinted against 13 and identical (0642e41750ef8bab), fold alignment verified, leak checks reproduce 13's numbers. Kaggle CPU, 56m 38s for all five seeds in one kernel, the encoder built once per fold and shared by every arm so the arms cannot differ through its inner split. The seed-42 arm reproduces row 38's saved vector BIT-IDENTICALLY, max |diff| 0.0 across all 691,369 predictions, and its CV to -1.46e-07. A seed-live check ran before the sweep and confirmed a different seed moves the predictions, without which five silently identical arms would have read as a spectacular stability result rather than as a bug. Not submitted on its own; exists to size the seed spread and to be stacked. THE POINT OF THE SWEEP, AND IT RESOLVES ROW 38 UPWARD: the five XGBoost seeds span 4.87e-05 against CatBoost's 1.32e-05 (rows 28 to 31) and LightGBM target-encoded's 6.00e-05 (rows 20 to 23), so XGBoost is about 3.7x less stable than CatBoost under its own randomness and slightly more stable than LightGBM. The family mean is 0.967118 over five seeds, which is +0.000355 above the LightGBM target-encoded family at 23.6 standard errors and +0.000197 above the CatBoost family. CORRECTED 2026-08-20: this row first recorded +0.000336 at 35.1 se and +0.000203, which compared the XGBoost FAMILY MEAN against a SINGLE SEED of each other learner (rows 17 and 26) and used a one-sample standard error that treated the baseline as a known constant. Rows 28 to 31 had already established family against family as the convention here and 25_xgb_seeds.ipynb deviated from it. The corrected figures are family against family with a two-sample standard error, computed by writeup/writeup_numbers.py from the out-of-fold vectors independently of the notebook. The gap is larger than first reported and its significance is lower, because the honest error bar includes LightGBM's own seed spread. The conclusion is unchanged. Row 38 was logged as one seed and explicitly not as an improvement, on row 26's precedent; that caution is now discharged. XGBoost is the strongest single-model family on this representation, and the margin is not close."},{"id":41,"utc":"2026-08-19 22:35","name":"xgb_te_seed7","cv_mean":0.967132,"cv_std":0.000455,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 38's configuration at a different XGBoost seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Encoder fingerprinted against 13 and identical (0642e41750ef8bab), fold alignment verified, leak checks reproduce 13's numbers. Kaggle CPU, 56m 38s for all five seeds in one kernel, the encoder built once per fold and shared by every arm so the arms cannot differ through its inner split. The seed-42 arm reproduces row 38's saved vector BIT-IDENTICALLY, max |diff| 0.0 across all 691,369 predictions, and its CV to -1.46e-07. A seed-live check ran before the sweep and confirmed a different seed moves the predictions, without which five silently identical arms would have read as a spectacular stability result rather than as a bug. Not submitted on its own; exists to size the seed spread and to be stacked. THE POINT OF THE SWEEP, AND IT RESOLVES ROW 38 UPWARD: the five XGBoost seeds span 4.87e-05 against CatBoost's 1.32e-05 (rows 28 to 31) and LightGBM target-encoded's 6.00e-05 (rows 20 to 23), so XGBoost is about 3.7x less stable than CatBoost under its own randomness and slightly more stable than LightGBM. The family mean is 0.967118 over five seeds, which is +0.000355 above the LightGBM target-encoded family at 23.6 standard errors and +0.000197 above the CatBoost family. CORRECTED 2026-08-20: this row first recorded +0.000336 at 35.1 se and +0.000203, which compared the XGBoost FAMILY MEAN against a SINGLE SEED of each other learner (rows 17 and 26) and used a one-sample standard error that treated the baseline as a known constant. Rows 28 to 31 had already established family against family as the convention here and 25_xgb_seeds.ipynb deviated from it. The corrected figures are family against family with a two-sample standard error, computed by writeup/writeup_numbers.py from the out-of-fold vectors independently of the notebook. The gap is larger than first reported and its significance is lower, because the honest error bar includes LightGBM's own seed spread. The conclusion is unchanged. Row 38 was logged as one seed and explicitly not as an improvement, on row 26's precedent; that caution is now discharged. XGBoost is the strongest single-model family on this representation, and the margin is not close."},{"id":42,"utc":"2026-08-19 22:35","name":"xgb_te_seed2025","cv_mean":0.967099,"cv_std":0.000421,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 38's configuration at a different XGBoost seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Encoder fingerprinted against 13 and identical (0642e41750ef8bab), fold alignment verified, leak checks reproduce 13's numbers. Kaggle CPU, 56m 38s for all five seeds in one kernel, the encoder built once per fold and shared by every arm so the arms cannot differ through its inner split. The seed-42 arm reproduces row 38's saved vector BIT-IDENTICALLY, max |diff| 0.0 across all 691,369 predictions, and its CV to -1.46e-07. A seed-live check ran before the sweep and confirmed a different seed moves the predictions, without which five silently identical arms would have read as a spectacular stability result rather than as a bug. Not submitted on its own; exists to size the seed spread and to be stacked. THE POINT OF THE SWEEP, AND IT RESOLVES ROW 38 UPWARD: the five XGBoost seeds span 4.87e-05 against CatBoost's 1.32e-05 (rows 28 to 31) and LightGBM target-encoded's 6.00e-05 (rows 20 to 23), so XGBoost is about 3.7x less stable than CatBoost under its own randomness and slightly more stable than LightGBM. The family mean is 0.967118 over five seeds, which is +0.000355 above the LightGBM target-encoded family at 23.6 standard errors and +0.000197 above the CatBoost family. CORRECTED 2026-08-20: this row first recorded +0.000336 at 35.1 se and +0.000203, which compared the XGBoost FAMILY MEAN against a SINGLE SEED of each other learner (rows 17 and 26) and used a one-sample standard error that treated the baseline as a known constant. Rows 28 to 31 had already established family against family as the convention here and 25_xgb_seeds.ipynb deviated from it. The corrected figures are family against family with a two-sample standard error, computed by writeup/writeup_numbers.py from the out-of-fold vectors independently of the notebook. The gap is larger than first reported and its significance is lower, because the honest error bar includes LightGBM's own seed spread. The conclusion is unchanged. Row 38 was logged as one seed and explicitly not as an improvement, on row 26's precedent; that caution is now discharged. XGBoost is the strongest single-model family on this representation, and the margin is not close."},{"id":43,"utc":"2026-08-19 22:35","name":"xgb_te_seed13","cv_mean":0.967111,"cv_std":0.000446,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"row 38's configuration at a different XGBoost seed, everything else identical including the encoder's inner split, so only the model's stochasticity varies. Encoder fingerprinted against 13 and identical (0642e41750ef8bab), fold alignment verified, leak checks reproduce 13's numbers. Kaggle CPU, 56m 38s for all five seeds in one kernel, the encoder built once per fold and shared by every arm so the arms cannot differ through its inner split. The seed-42 arm reproduces row 38's saved vector BIT-IDENTICALLY, max |diff| 0.0 across all 691,369 predictions, and its CV to -1.46e-07. A seed-live check ran before the sweep and confirmed a different seed moves the predictions, without which five silently identical arms would have read as a spectacular stability result rather than as a bug. Not submitted on its own; exists to size the seed spread and to be stacked. THE POINT OF THE SWEEP, AND IT RESOLVES ROW 38 UPWARD: the five XGBoost seeds span 4.87e-05 against CatBoost's 1.32e-05 (rows 28 to 31) and LightGBM target-encoded's 6.00e-05 (rows 20 to 23), so XGBoost is about 3.7x less stable than CatBoost under its own randomness and slightly more stable than LightGBM. The family mean is 0.967118 over five seeds, which is +0.000355 above the LightGBM target-encoded family at 23.6 standard errors and +0.000197 above the CatBoost family. CORRECTED 2026-08-20: this row first recorded +0.000336 at 35.1 se and +0.000203, which compared the XGBoost FAMILY MEAN against a SINGLE SEED of each other learner (rows 17 and 26) and used a one-sample standard error that treated the baseline as a known constant. Rows 28 to 31 had already established family against family as the convention here and 25_xgb_seeds.ipynb deviated from it. The corrected figures are family against family with a two-sample standard error, computed by writeup/writeup_numbers.py from the out-of-fold vectors independently of the notebook. The gap is larger than first reported and its significance is lower, because the honest error bar includes LightGBM's own seed spread. The conclusion is unchanged. Row 38 was logged as one seed and explicitly not as an improvement, on row 26's precedent; that caution is now discharged. XGBoost is the strongest single-model family on this representation, and the margin is not close."},{"id":44,"utc":"2026-08-19 22:52","name":"stack_logit_29_oof","cv_mean":0.967925,"cv_std":0.000428,"folds":5,"lb_public":0.96924,"lb_private":null,"submitted":"yes","notes":"Row 39's stack with the four extra XGBoost seeds added. One variable against row 39: same combiner, same C, same fold-wise protocol, same folds; only the member set changes, 25 to 29. Row 39 is REFIT in the same run and reproduces its recorded CV to -2.07e-07. Paired vs row 39 +0.000052, sd 0.000013, 5/5 folds, t(4)=9.16. PASSES THE PRE-REGISTERED GATE, at 4 of 5 folds and +0.00005, but only just: +0.000052 against a floor of +0.000050. Read it as the smallest gain this repo has ever accepted rather than as a comfortable pass. THE PREDICTION IN THE NOTEBOOK HEADER WAS RECORDED BEFORE THE RUN AND WAS NARROWLY LOW. It predicted +0.000014 to +0.000050 by a splitting mechanism, reasoning from row 32's equivalent CatBoost experiment (+0.000014) scaled up by XGBoost's 3.7x larger seed spread. The actual is +0.000052, just above the top of the range, so the magnitude was right and the bound was slightly tight. THE MECHANISM IS SPLITTING PLUS A REAL ADDITION, UNLIKE ROW 32. The five XGBoost coefficients sum to +0.4673 against xgb_te's +0.2751 as a lone member, so the five are worth +0.1922 more than the one. Row 32's CatBoost equivalent was +0.0504. That ratio tracks the seed spreads, 4.87e-05 here against 1.32e-05 there, which is what seed averaging is supposed to do and is the first time in this repo it has behaved textbook. WHAT IT EXPOSES, AND IT IS THE MORE INTERESTING RESULT: the five LightGBM target-encoded seeds have collapsed. te42 is now -0.0032 and te7 is -0.0089, both NEGATIVE, with te13 +0.0081, te2024 +0.0033 and te2025 +0.0032, so all five together carry about +0.0025 against roughly +0.081 each in row 34. XGBoost has taken essentially all of their weight. Whether they still earn a place is a removal experiment and a separate ledger row. Largest fold-to-fold coefficient sd 0.0326, on bag2024, the same redundant member identified since row 25. CV 0.967925 is the best in this file. Spearman against row 39 is 0.9998940. NOT SUBMITTED, on the precedent rows 25, 34 and 39 set. Run locally by nbconvert --execute on a clean kernel, notebooks/27_stack_29.ipynb. SUBMITTED 2026-08-20, LB 0.96924, THE BEST IN THIS REPO, against row 32's 0.96902. CV moved +0.000161 and LB moved +0.000220, same direction, so CV and LB now agree on 9 of 9 submissions. The CV-to-LB offset is +0.001315, against +0.001256 for row 32 and +0.001305 for row 24, so the offset is stable to about 6e-05 across the three stack submissions. The LB gain was predicted at +0.00017 before the score returned, from that offset, and came in at +0.00022. WHY THIS ONE WAS SUBMITTED WHEN ROWS 25, 34 AND 39 WERE HELD BACK, and it is a correction to the reasoning that held them: Spearman against the submitted row 32 is 0.999261, not the 0.9998-and-up figures those decisions were made on. Those compared each stack to its immediate predecessor rather than to the thing actually on the leaderboard, and the differences had accumulated. Row 27 was called 'a materially different ordering' at 0.9994618, so this clears that bar. The stronger reason is that XGBoost is a model family that had never been LB-validated and now carries the largest weight block in the stack, so this submission tests whether a newly dominant component preserves the CV/LB relationship. It does. A STANDING ERROR IN THIS FILE, CORRECTED: the 'not worth a slot' argument behind rows 25, 34 and 39 assumed submissions were scarce. The daily quota is 10 and 9 remained after this one. Slots were never the binding constraint; the real argument was only ever about not re-measuring an ordering the leaderboard cannot resolve, which is a narrower claim than the one that was made. Rank 602 of 2369, top 25.4 percent, against 590 of 2229 and top 26.5 percent before. The rank number rose while the percentile improved, because the field grew by 140 entrants. THE GAP TO TOP 50 NARROWED FOR THE FIRST TIME IN THIS COMPETITION: 0.00193, against 0.00197 on 2026-08-12 and 0.00212 on 2026-08-18."},{"id":45,"utc":"2026-08-19 23:05","name":"stack_logit_24_pruned","cv_mean":0.967929,"cv_std":0.000429,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"Row 44's stack with the five LightGBM target-encoded seeds REMOVED, 29 members to 24. One variable against row 44: the member set; same combiner, same C, same fold-wise protocol, same folds. Row 44 is refit in the same run and reproduces its recorded CV to +1.23e-07. The first experiment in this competition that makes the stack smaller. CV 0.967929 against row 44's 0.967925, paired +0.000004, sd 0.000002, 5/5 folds. That is four millionths, against an equivalence band of 5.2e-05 set before the run at row 44's own accepted gain. Removing five of twenty-nine members changes nothing measurable. THE PRE-REGISTERED HONESTY CHECK FIRED AND THE VERDICT IS DISCARD. The removal set was chosen by reading row 44's coefficients off this same out-of-fold matrix, which is selection on the validation set, the error this repo refused to commit when it kept seed 2024 in the raw-feature blend. The notebook pre-registered a check that the drop be supported by training-fold information alone, and it failed: the largest coefficient a dropped member reached in a training-fold fit was +0.0193, above the +0.0051 threshold the check compared it against. THE CHECK WAS MIS-SPECIFIED AND THAT IS MY ERROR, NOT A PROPERTY OF THE DATA. It compared a MAX over five folds for the dropped members against a MEAN over five folds for the kept members, which is not like for like, and a single-fold maximum will exceed a mean almost by construction. The verdict stands as written rather than being rerun with a friendlier rule, because moving a pre-registered bar after seeing the number is the failure the bar exists to prevent. THE SUBSTANTIVE CONCLUSION SURVIVES ANYWAY, FOR A REASON INDEPENDENT OF THE CHECK. Selection bias here can only inflate the pruned stack, since the members were chosen for looking useless on these rows. The inflated estimate is +0.000004, so the true effect of removal is approximately zero and at worst slightly negative, far inside the band either way. The five dropped coefficients oscillate in sign across folds (te42 runs +0.0145 to -0.0090, te13 +0.0193 to -0.0031), which is noise around zero rather than suppressed signal. XGBoost has taken their role. WHAT IS ACTUALLY DONE WITH IT: nothing. Row 44 stays the selection candidate. Spearman between the two is 0.9999992, so they are the same submission, and simplicity alone does not justify acting against a pre-registered discard. Run locally by nbconvert --execute on a clean kernel, notebooks/28_stack_prune.ipynb."},{"id":46,"utc":"2026-08-20 03:49","name":"xgb_smooth1","cv_mean":0.967122,"cv_std":0.00044,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"One variable against row 38: the encoder's smoothing constant. Same learner, same folds, same seed 42, same budget, same N_INNER=5; only SMOOTH moves. The encoder source is untouched and still fingerprints 0642e41750ef8bab, because _stats reads SMOOTH from module globals, so each arm rebinds the constant and calls the same function. A live-constant check ran before the sweep and confirmed SMOOTH reaches the encoder (0.366 largest change in an encoded column between 10 and 100); without it five identical arms would have produced a perfectly flat curve, which is exactly the result the header predicted, and the coincidence would have been indistinguishable from success. Kaggle CPU, 25 encoder builds. Leak checks PASS, fold alignment verified, determinism OK, and the SMOOTH=10 arm reproduces row 38 to -1.46e-07. Not submitted, not carried forward. THE POINT OF THE SWEEP: SMOOTH was written into cell 1 of 13_target_encoding.ipynb on 2026-08-11, chosen once, and 22 of the 29 members of row 44 inherited it without it ever being varied. It was the last untested constant in the pipeline. NULL, AND THE PREDICTION HELD. The curve is flat from 1 to 10, where 1.0 and 5.0 sit +0.000023 and +0.000017 above the baseline on 3 of 5 folds, which is a coin flip, and it falls above 10, at -0.000062 (0/5) for 25 and -0.000219 (0/5) for 100. SMOOTH=10 sits on the flat part. The header predicted this from the arithmetic: at roughly 500 rows per level the shrinkage toward the prior is about 2 percent, so the constant is close to inert by construction, and the grid ran to 100 to confirm the flat part was actually being left before calling it flat. WHAT IT CONFIRMS BEYOND ITSELF: this is the fifth reopening measured in this repo and the pattern recorded after num_leaves now has five points. The three that paid changed the LEARNER or its input (CatBoost row 26, the neural model row 33, XGBoost row 38). The two that returned nothing changed a HYPERPARAMETER of a component already fitted to the representation (num_leaves rows 35 to 37, SMOOTH here). A change of representation revalues learners, not their knobs."},{"id":47,"utc":"2026-08-20 03:49","name":"xgb_smooth5","cv_mean":0.967116,"cv_std":0.000443,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"One variable against row 38: the encoder's smoothing constant. Same learner, same folds, same seed 42, same budget, same N_INNER=5; only SMOOTH moves. The encoder source is untouched and still fingerprints 0642e41750ef8bab, because _stats reads SMOOTH from module globals, so each arm rebinds the constant and calls the same function. A live-constant check ran before the sweep and confirmed SMOOTH reaches the encoder (0.366 largest change in an encoded column between 10 and 100); without it five identical arms would have produced a perfectly flat curve, which is exactly the result the header predicted, and the coincidence would have been indistinguishable from success. Kaggle CPU, 25 encoder builds. Leak checks PASS, fold alignment verified, determinism OK, and the SMOOTH=10 arm reproduces row 38 to -1.46e-07. Not submitted, not carried forward. THE POINT OF THE SWEEP: SMOOTH was written into cell 1 of 13_target_encoding.ipynb on 2026-08-11, chosen once, and 22 of the 29 members of row 44 inherited it without it ever being varied. It was the last untested constant in the pipeline. NULL, AND THE PREDICTION HELD. The curve is flat from 1 to 10, where 1.0 and 5.0 sit +0.000023 and +0.000017 above the baseline on 3 of 5 folds, which is a coin flip, and it falls above 10, at -0.000062 (0/5) for 25 and -0.000219 (0/5) for 100. SMOOTH=10 sits on the flat part. The header predicted this from the arithmetic: at roughly 500 rows per level the shrinkage toward the prior is about 2 percent, so the constant is close to inert by construction, and the grid ran to 100 to confirm the flat part was actually being left before calling it flat. WHAT IT CONFIRMS BEYOND ITSELF: this is the fifth reopening measured in this repo and the pattern recorded after num_leaves now has five points. The three that paid changed the LEARNER or its input (CatBoost row 26, the neural model row 33, XGBoost row 38). The two that returned nothing changed a HYPERPARAMETER of a component already fitted to the representation (num_leaves rows 35 to 37, SMOOTH here). A change of representation revalues learners, not their knobs."},{"id":48,"utc":"2026-08-20 03:49","name":"xgb_smooth25","cv_mean":0.967037,"cv_std":0.000418,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"One variable against row 38: the encoder's smoothing constant. Same learner, same folds, same seed 42, same budget, same N_INNER=5; only SMOOTH moves. The encoder source is untouched and still fingerprints 0642e41750ef8bab, because _stats reads SMOOTH from module globals, so each arm rebinds the constant and calls the same function. A live-constant check ran before the sweep and confirmed SMOOTH reaches the encoder (0.366 largest change in an encoded column between 10 and 100); without it five identical arms would have produced a perfectly flat curve, which is exactly the result the header predicted, and the coincidence would have been indistinguishable from success. Kaggle CPU, 25 encoder builds. Leak checks PASS, fold alignment verified, determinism OK, and the SMOOTH=10 arm reproduces row 38 to -1.46e-07. Not submitted, not carried forward. THE POINT OF THE SWEEP: SMOOTH was written into cell 1 of 13_target_encoding.ipynb on 2026-08-11, chosen once, and 22 of the 29 members of row 44 inherited it without it ever being varied. It was the last untested constant in the pipeline. NULL, AND THE PREDICTION HELD. The curve is flat from 1 to 10, where 1.0 and 5.0 sit +0.000023 and +0.000017 above the baseline on 3 of 5 folds, which is a coin flip, and it falls above 10, at -0.000062 (0/5) for 25 and -0.000219 (0/5) for 100. SMOOTH=10 sits on the flat part. The header predicted this from the arithmetic: at roughly 500 rows per level the shrinkage toward the prior is about 2 percent, so the constant is close to inert by construction, and the grid ran to 100 to confirm the flat part was actually being left before calling it flat. WHAT IT CONFIRMS BEYOND ITSELF: this is the fifth reopening measured in this repo and the pattern recorded after num_leaves now has five points. The three that paid changed the LEARNER or its input (CatBoost row 26, the neural model row 33, XGBoost row 38). The two that returned nothing changed a HYPERPARAMETER of a component already fitted to the representation (num_leaves rows 35 to 37, SMOOTH here). A change of representation revalues learners, not their knobs."},{"id":49,"utc":"2026-08-20 03:49","name":"xgb_smooth100","cv_mean":0.96688,"cv_std":0.000426,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"One variable against row 38: the encoder's smoothing constant. Same learner, same folds, same seed 42, same budget, same N_INNER=5; only SMOOTH moves. The encoder source is untouched and still fingerprints 0642e41750ef8bab, because _stats reads SMOOTH from module globals, so each arm rebinds the constant and calls the same function. A live-constant check ran before the sweep and confirmed SMOOTH reaches the encoder (0.366 largest change in an encoded column between 10 and 100); without it five identical arms would have produced a perfectly flat curve, which is exactly the result the header predicted, and the coincidence would have been indistinguishable from success. Kaggle CPU, 25 encoder builds. Leak checks PASS, fold alignment verified, determinism OK, and the SMOOTH=10 arm reproduces row 38 to -1.46e-07. Not submitted, not carried forward. THE POINT OF THE SWEEP: SMOOTH was written into cell 1 of 13_target_encoding.ipynb on 2026-08-11, chosen once, and 22 of the 29 members of row 44 inherited it without it ever being varied. It was the last untested constant in the pipeline. NULL, AND THE PREDICTION HELD. The curve is flat from 1 to 10, where 1.0 and 5.0 sit +0.000023 and +0.000017 above the baseline on 3 of 5 folds, which is a coin flip, and it falls above 10, at -0.000062 (0/5) for 25 and -0.000219 (0/5) for 100. SMOOTH=10 sits on the flat part. The header predicted this from the arithmetic: at roughly 500 rows per level the shrinkage toward the prior is about 2 percent, so the constant is close to inert by construction, and the grid ran to 100 to confirm the flat part was actually being left before calling it flat. WHAT IT CONFIRMS BEYOND ITSELF: this is the fifth reopening measured in this repo and the pattern recorded after num_leaves now has five points. The three that paid changed the LEARNER or its input (CatBoost row 26, the neural model row 33, XGBoost row 38). The two that returned nothing changed a HYPERPARAMETER of a component already fitted to the representation (num_leaves rows 35 to 37, SMOOTH here). A change of representation revalues learners, not their knobs."},{"id":50,"utc":"2026-08-20 16:34","name":"xgb_inner2","cv_mean":0.966937,"cv_std":0.000435,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"Arm N_INNER=2, paired -0.000162 vs the base arm, sd 0.000087, 0/5 folds. The only arm that loses on every fold, so the curve does have a falling side and the flat part is genuinely being left. One variable against row 38 (xgb_te, 0.967099): the target encoder's inner split count N_INNER, the number of folds used to build TRAINING-row encodings. Same learner, same outer folds, same seed, same budget, same SMOOTH=10.0, same encoder source (fingerprint 0642e41750ef8bab, matches 13). The last constant in this pipeline that had never been varied: 13_target_encoding.ipynb set it to 5 on 2026-08-11 and 22 of row 44's 29 members inherited it. THE ARM AT N_INNER=5 REPRODUCES ROW 38 AT -1.46e-07, which is the tightest reproduction in this repo. Fold alignment verified against sha ec282b0968059676, leak checks pass, determinism exact. THE POSITIVE CONTROL IS TWO-SIDED AND BOTH SIDES FIRED. N_INNER is read only by the inner out-of-fold pass, so it must move training-row encodings and must leave validation-row encodings bit-identical. Measured before any model trained: train moved 1.637e-01, validation moved exactly 0. Notebook 29's control could only check the first of those, because SMOOTH reaches every column. SWEEP RESULT, NULL. Arms 2/3/5/10/20 give 0.966937/0.967016/0.967099/0.967107/0.967096. Flat at and above 5, falling below it. The best arm is N_INNER=10 at +0.000009 paired, 2/5 folds, which is not paired-significant and is 11x under the +0.00010 gate floor pre-registered in the header. THE HEADER'S ALTERNATIVE MECHANISM WAS PREDICTED AND REFUTED. Training rows are encoded from (N_INNER-1)/N_INNER of the fold while validation rows are encoded from all of it, a real train/serve mismatch that a larger N_INNER narrows, so the header predicted a curve rising monotonically toward 20. It does not rise: 10 is +0.000009 and 20 is -0.000003, both inside noise. The mismatch is real and worth nothing, which is the same shape as the decimal lattice in row 19. SIXTH POINT ON THE RULE IN NOTES.md, and it is the second knob to return nothing on this representation after num_leaves and SMOOTH. Three learner reopenings paid, three hyperparameter sweeps did not. Kaggle Save and Run All, vyask21/smartphone-addiction-inner version 1, 36m 28s, notebooks/30_inner_splits.ipynb."},{"id":51,"utc":"2026-08-20 16:34","name":"xgb_inner3","cv_mean":0.967016,"cv_std":0.000428,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"Arm N_INNER=3, paired -0.000083 vs the base arm, sd 0.000075, 1/5 folds. One variable against row 38 (xgb_te, 0.967099): the target encoder's inner split count N_INNER, the number of folds used to build TRAINING-row encodings. Same learner, same outer folds, same seed, same budget, same SMOOTH=10.0, same encoder source (fingerprint 0642e41750ef8bab, matches 13). The last constant in this pipeline that had never been varied: 13_target_encoding.ipynb set it to 5 on 2026-08-11 and 22 of row 44's 29 members inherited it. THE ARM AT N_INNER=5 REPRODUCES ROW 38 AT -1.46e-07, which is the tightest reproduction in this repo. Fold alignment verified against sha ec282b0968059676, leak checks pass, determinism exact. THE POSITIVE CONTROL IS TWO-SIDED AND BOTH SIDES FIRED. N_INNER is read only by the inner out-of-fold pass, so it must move training-row encodings and must leave validation-row encodings bit-identical. Measured before any model trained: train moved 1.637e-01, validation moved exactly 0. Notebook 29's control could only check the first of those, because SMOOTH reaches every column. SWEEP RESULT, NULL. Arms 2/3/5/10/20 give 0.966937/0.967016/0.967099/0.967107/0.967096. Flat at and above 5, falling below it. The best arm is N_INNER=10 at +0.000009 paired, 2/5 folds, which is not paired-significant and is 11x under the +0.00010 gate floor pre-registered in the header. THE HEADER'S ALTERNATIVE MECHANISM WAS PREDICTED AND REFUTED. Training rows are encoded from (N_INNER-1)/N_INNER of the fold while validation rows are encoded from all of it, a real train/serve mismatch that a larger N_INNER narrows, so the header predicted a curve rising monotonically toward 20. It does not rise: 10 is +0.000009 and 20 is -0.000003, both inside noise. The mismatch is real and worth nothing, which is the same shape as the decimal lattice in row 19. SIXTH POINT ON THE RULE IN NOTES.md, and it is the second knob to return nothing on this representation after num_leaves and SMOOTH. Three learner reopenings paid, three hyperparameter sweeps did not. Kaggle Save and Run All, vyask21/smartphone-addiction-inner version 1, 36m 28s, notebooks/30_inner_splits.ipynb."},{"id":52,"utc":"2026-08-20 16:34","name":"xgb_inner5","cv_mean":0.967099,"cv_std":0.000414,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"Arm N_INNER=5, the value 13 chose and the value row 38 used, so this arm is the reproduction check rather than a treatment. Reproduces row 38 at -1.46e-07. One variable against row 38 (xgb_te, 0.967099): the target encoder's inner split count N_INNER, the number of folds used to build TRAINING-row encodings. Same learner, same outer folds, same seed, same budget, same SMOOTH=10.0, same encoder source (fingerprint 0642e41750ef8bab, matches 13). The last constant in this pipeline that had never been varied: 13_target_encoding.ipynb set it to 5 on 2026-08-11 and 22 of row 44's 29 members inherited it. THE ARM AT N_INNER=5 REPRODUCES ROW 38 AT -1.46e-07, which is the tightest reproduction in this repo. Fold alignment verified against sha ec282b0968059676, leak checks pass, determinism exact. THE POSITIVE CONTROL IS TWO-SIDED AND BOTH SIDES FIRED. N_INNER is read only by the inner out-of-fold pass, so it must move training-row encodings and must leave validation-row encodings bit-identical. Measured before any model trained: train moved 1.637e-01, validation moved exactly 0. Notebook 29's control could only check the first of those, because SMOOTH reaches every column. SWEEP RESULT, NULL. Arms 2/3/5/10/20 give 0.966937/0.967016/0.967099/0.967107/0.967096. Flat at and above 5, falling below it. The best arm is N_INNER=10 at +0.000009 paired, 2/5 folds, which is not paired-significant and is 11x under the +0.00010 gate floor pre-registered in the header. THE HEADER'S ALTERNATIVE MECHANISM WAS PREDICTED AND REFUTED. Training rows are encoded from (N_INNER-1)/N_INNER of the fold while validation rows are encoded from all of it, a real train/serve mismatch that a larger N_INNER narrows, so the header predicted a curve rising monotonically toward 20. It does not rise: 10 is +0.000009 and 20 is -0.000003, both inside noise. The mismatch is real and worth nothing, which is the same shape as the decimal lattice in row 19. SIXTH POINT ON THE RULE IN NOTES.md, and it is the second knob to return nothing on this representation after num_leaves and SMOOTH. Three learner reopenings paid, three hyperparameter sweeps did not. Kaggle Save and Run All, vyask21/smartphone-addiction-inner version 1, 36m 28s, notebooks/30_inner_splits.ipynb."},{"id":53,"utc":"2026-08-20 16:34","name":"xgb_inner10","cv_mean":0.967107,"cv_std":0.000427,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"Arm N_INNER=10, paired +0.000009 vs the base arm, sd 0.000047, 2/5 folds. The best arm in the sweep and it is nowhere near the gate. One variable against row 38 (xgb_te, 0.967099): the target encoder's inner split count N_INNER, the number of folds used to build TRAINING-row encodings. Same learner, same outer folds, same seed, same budget, same SMOOTH=10.0, same encoder source (fingerprint 0642e41750ef8bab, matches 13). The last constant in this pipeline that had never been varied: 13_target_encoding.ipynb set it to 5 on 2026-08-11 and 22 of row 44's 29 members inherited it. THE ARM AT N_INNER=5 REPRODUCES ROW 38 AT -1.46e-07, which is the tightest reproduction in this repo. Fold alignment verified against sha ec282b0968059676, leak checks pass, determinism exact. THE POSITIVE CONTROL IS TWO-SIDED AND BOTH SIDES FIRED. N_INNER is read only by the inner out-of-fold pass, so it must move training-row encodings and must leave validation-row encodings bit-identical. Measured before any model trained: train moved 1.637e-01, validation moved exactly 0. Notebook 29's control could only check the first of those, because SMOOTH reaches every column. SWEEP RESULT, NULL. Arms 2/3/5/10/20 give 0.966937/0.967016/0.967099/0.967107/0.967096. Flat at and above 5, falling below it. The best arm is N_INNER=10 at +0.000009 paired, 2/5 folds, which is not paired-significant and is 11x under the +0.00010 gate floor pre-registered in the header. THE HEADER'S ALTERNATIVE MECHANISM WAS PREDICTED AND REFUTED. Training rows are encoded from (N_INNER-1)/N_INNER of the fold while validation rows are encoded from all of it, a real train/serve mismatch that a larger N_INNER narrows, so the header predicted a curve rising monotonically toward 20. It does not rise: 10 is +0.000009 and 20 is -0.000003, both inside noise. The mismatch is real and worth nothing, which is the same shape as the decimal lattice in row 19. SIXTH POINT ON THE RULE IN NOTES.md, and it is the second knob to return nothing on this representation after num_leaves and SMOOTH. Three learner reopenings paid, three hyperparameter sweeps did not. Kaggle Save and Run All, vyask21/smartphone-addiction-inner version 1, 36m 28s, notebooks/30_inner_splits.ipynb."},{"id":54,"utc":"2026-08-20 16:34","name":"xgb_inner20","cv_mean":0.967096,"cv_std":0.000444,"folds":5,"lb_public":null,"lb_private":null,"submitted":"no","notes":"Arm N_INNER=20, paired -0.000003 vs the base arm, sd 0.000134, 3/5 folds. Four times the encoder cost of the base arm for nothing, which is the practical reading of this sweep. One variable against row 38 (xgb_te, 0.967099): the target encoder's inner split count N_INNER, the number of folds used to build TRAINING-row encodings. Same learner, same outer folds, same seed, same budget, same SMOOTH=10.0, same encoder source (fingerprint 0642e41750ef8bab, matches 13). The last constant in this pipeline that had never been varied: 13_target_encoding.ipynb set it to 5 on 2026-08-11 and 22 of row 44's 29 members inherited it. THE ARM AT N_INNER=5 REPRODUCES ROW 38 AT -1.46e-07, which is the tightest reproduction in this repo. Fold alignment verified against sha ec282b0968059676, leak checks pass, determinism exact. THE POSITIVE CONTROL IS TWO-SIDED AND BOTH SIDES FIRED. N_INNER is read only by the inner out-of-fold pass, so it must move training-row encodings and must leave validation-row encodings bit-identical. Measured before any model trained: train moved 1.637e-01, validation moved exactly 0. Notebook 29's control could only check the first of those, because SMOOTH reaches every column. SWEEP RESULT, NULL. Arms 2/3/5/10/20 give 0.966937/0.967016/0.967099/0.967107/0.967096. Flat at and above 5, falling below it. The best arm is N_INNER=10 at +0.000009 paired, 2/5 folds, which is not paired-significant and is 11x under the +0.00010 gate floor pre-registered in the header. THE HEADER'S ALTERNATIVE MECHANISM WAS PREDICTED AND REFUTED. Training rows are encoded from (N_INNER-1)/N_INNER of the fold while validation rows are encoded from all of it, a real train/serve mismatch that a larger N_INNER narrows, so the header predicted a curve rising monotonically toward 20. It does not rise: 10 is +0.000009 and 20 is -0.000003, both inside noise. The mismatch is real and worth nothing, which is the same shape as the decimal lattice in row 19. SIXTH POINT ON THE RULE IN NOTES.md, and it is the second knob to return nothing on this representation after num_leaves and SMOOTH. Three learner reopenings paid, three hyperparameter sweeps did not. Kaggle Save and Run All, vyask21/smartphone-addiction-inner version 1, 36m 28s, notebooks/30_inner_splits.ipynb."}]}''')

print(f"{len(D['pairs'])} model pairs, {len(D['ledger'])} ledger rows, "
      f"{len(D['missingness'])} features")


### Chart styling

Two colours, checked for colour-vision deficiency separation against a white
background rather than picked by eye. Everything else is recessive on purpose:
hairline grid, no top or right spine, labels in grey so the marks carry the ink.

In [ ]:
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, SECOND, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#ffffff"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": SECOND, "axes.edgecolor": AXIS,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "figure.dpi": 120,
})


def finish(ax, title, sub=None):
    """Title and subtitle stacked above the axes.

    Placed by hand rather than with set_title, which collides with a subtitle
    drawn just above the axes.
    """
    n = sub.count("\n") + 1 if sub else 0
    ax.text(0, 1.06 + 0.055 * n, title, transform=ax.transAxes, color=INK,
            fontsize=12, fontweight="bold", va="bottom")
    if sub:
        ax.text(0, 1.03, sub, transform=ax.transAxes, color=SECOND,
                fontsize=9.5, va="bottom", linespacing=1.4)
    ax.set_axisbelow(True)


print("styled")

## First: the metric and the validation scheme

Before looking at a single feature. The metric is ROC AUC on a predicted
probability, which decides three things at once: the loss is log loss, the
predictions never need calibrating because AUC only reads their order, and rank
averaging is the natural way to combine models.

The split is **5-fold stratified, shuffled, seed 42**, and it never changed. There
are no groups, no repeated entities and no time index in this data, so stratifying
on the target is the whole requirement. Every number in this notebook comes from
that one split, which is what makes them comparable to each other.

The thing worth copying is not the split. It is that **the split was fixed and the
fold assignment checksummed** before any modelling. Every out-of-fold vector the
repo saved reproduces its recorded fold-mean to within 5e-7, so any two of them can
be blended and compared row for row, months apart. Half of what follows would have
been unmeasurable without that.

## The data is synthetic, and that decided the feature strategy

Playground Series data is generated from an original public dataset. Community
forensic work on this one (credit to Busya PRIME and broccoli beef in the
competition discussion) reported that the original is close to a lookup table:
`addicted` is 1 when `daily_screen_time_hours > 8` or `social_media_hours > 4`.

I did not verify those rules myself, so treat them as reported rather than
established. They score about **0.9888 AUC on the original data and about
0.835 on the synthetic data we are actually scored on.**

That single pair of numbers rewrote my plan. My feature list was a stack of
threshold and ratio features aimed squarely at recovering those rules, and an
untuned LightGBM already scores 0.9549, well above the 0.835 ceiling the rules
themselves reach here. The generator has smeared the decision boundary into
something smooth, and encoding the original rules would have been a step backwards.

**Read the discussion tab before writing features.** It cost me twenty minutes and
saved a week of the most obvious possible dead end.

## My top-priority idea, killed in ninety seconds

Every feature in this dataset has missing values, between 4% and 20%. That is a
conspicuous amount of structure, and missingness indicators were the first thing on
my list, the reasoning being that whether someone declined to report their screen
time might say more than the number itself.

It is a testable claim and it costs one pass over the data, so I tested it before
spending a training run: for each feature, is the target rate different when that
feature is missing?

In [ ]:
mi = sorted(D["missingness"], key=lambda m: m["lift"])
names = [m["feature"] for m in mi]
lift = np.array([m["lift"] for m in mi])
# Each feature's own standard error, recovered from its z. A single shared band
# would have to be drawn at the widest feature's error, which would understate the
# certainty on the other eleven and flatter the conclusion.
se = np.array([abs(m["lift"] / m["z"]) if m["z"] else np.nan for m in mi])

fig, ax = plt.subplots(figsize=(9, 5))
yy = np.arange(len(names))
ax.axvline(0, color=AXIS, lw=1.2, zorder=1)
ax.errorbar(lift, yy, xerr=2 * se, fmt="o", color=BLUE, ms=8, mec=SURFACE,
            mew=1.5, ecolor=MUTED, elinewidth=1.4, capsize=3.5, capthick=1.4,
            zorder=3, label="±2 standard errors")
ax.set_yticks(yy)
ax.set_yticklabels(names, fontsize=9.5, color=SECOND)
ax.set_xlabel("target rate when the feature is missing, minus when it is present")
ax.legend(frameon=False, loc="lower right", labelcolor=SECOND)
ax.grid(axis="y", visible=False)
finish(ax, "Missingness carries no signal about the target",
       "Eleven of twelve intervals cross zero. The largest |z| is 2.24, against the\n"
       "2.4 you would expect from pure noise across twelve tests.")
plt.tight_layout()
plt.show()

print(f"largest |z|: {max(abs(m['z']) for m in mi):.2f}")

Not one feature clears the bar. The largest |z| across twelve tests is **2.24**,
and the largest value you would *expect* from twelve draws of pure noise is about
2.4. `app_opens_per_day` is the one interval that excludes zero, which is what one
in twelve looks like when nothing is going on.

The missingness was injected at random with respect to the target. LightGBM already
routes NaN natively, so twelve indicator columns would have added twelve columns of
nothing.

**This is the part I would most like people to copy.** The idea was wrong, and being
wrong cost ninety seconds instead of a training run and a submission, because it was
phrased as a measurable claim before it was phrased as a feature.

## Adversarial validation: mild, diffuse, not actionable

Train a classifier to tell train rows from test rows. If it succeeds, the two sets
differ and your CV is measuring the wrong distribution.

It scored **AUC 0.562868 +/- 0.000792**. Real but mild, and importance was spread
almost evenly across all eight numeric features at 10-11% each. A concentrated shift
gives you a feature to drop. A diffuse one gives you nothing to act on, so I noted
it and moved on rather than reaching for fold weighting.

Worth running anyway: it is ten minutes, and had it come back at 0.75 the entire
validation scheme would have needed rethinking before anything else was worth doing.

## The model, and what actually moved it

LightGBM, native categorical handling, native NaN handling, five-fold stratified CV
throughout.

| change | CV | gain |
|---|---|---|
| untuned defaults, 100 trees | 0.954947 | anchor |
| 1000 trees | 0.962141 | +0.007194 |
| lr 0.05, 2000 trees | 0.963210 | +0.001069 |
| bagging 0.8/0.8, 5 seeds, rank averaged | 0.963880 | +0.000670 |
| target and frequency encoding, all 12 columns | 0.966782 | +0.002902 |
| logistic stack over 18 models | 0.967665 | +0.000883 |

The first row is the embarrassing one and it is the most useful. The untuned baseline
was **badly underfit**, and 80% of everything I gained in the first three weeks was
recovered by asking for more trees. Fit the capacity before doing anything clever.

Tuning stopped after the learning rate. The step from lr 0.05 to 0.03 was +0.000064
at 60% more runtime, which is nothing, and the curve had visibly flattened.

The fifth row is a different kind of change from the four above it and it gets its
own section below.

In [ ]:
# Ledger row 9, retrained here so a row reproduces in front of you rather than
# being asserted. Not the final model: it is the raw-feature configuration, chosen
# because it is the one every comparison in this notebook is measured against. This is the slow cell: five folds of
# 2000 trees over 691k rows. Measured at 4.9 minutes on a Kaggle CPU kernel,
# about 56 seconds per fold; slower on fewer cores.
# Set to False to skip and just read the recorded numbers.
RUN_TRAINING = True

RECORDED_SEED42_CV = 0.963471   # experiments.csv row 9

# Do not expect this to reproduce exactly, and do not read a small difference as a bug.
# LightGBM's `deterministic=True` pins the result for a GIVEN THREAD COUNT, not across
# thread counts, because the flag fixes the order in which per-thread gradient sums are
# reduced and that order depends on how many threads there are. Measured on one fold of
# this exact config on the machine the ledger was produced on: n_jobs=6 reproduces the
# recorded value to 3.6e-7, while n_jobs=-1 (8 threads there) lands 3.0e-5 away, and
# both repeat to ~3e-7 across runs. Kaggle's core count is a third thing again.
#
# 1e-4 is the honest bar. It is well below the smallest gain in the ledger (+0.000238)
# and far below the +0.0089 the whole competition was worth, so nothing here rests on
# the fourth decimal of a single run.
TOLERANCE = 1e-4

PARAMS = dict(
    objective="binary", metric="auc", learning_rate=0.05, n_estimators=2000,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1,
    # Determinism. Without these two, two runs of the same config differ in the
    # fourth decimal, which is the same size as most of the gains being chased.
    deterministic=True, force_row_wise=True,
)

if RUN_TRAINING:
    import lightgbm as lgb
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedKFold

    from pathlib import Path

    def find_train():
        kag = Path("/kaggle/input")
        # Recursive: Kaggle mounts competitions under
        # /kaggle/input/competitions/<slug>/, not /kaggle/input/<slug>/.
        if kag.exists():
            hits = sorted(kag.rglob("train.csv"))
            if hits:
                return hits[0]
        for b in [Path.cwd(), *Path.cwd().parents]:
            p = b / "data" / "raw" / "train.csv"
            if p.exists():
                return p
        raise FileNotFoundError("train.csv not found")

    train = pd.read_csv(find_train())
    TARGET = "addicted_label"
    CAT = ["gender", "stress_level", "academic_work_impact"]
    feats = [c for c in train.columns if c not in ("id", TARGET)]
    X = train[feats].copy()
    for c in CAT:
        X[c] = X[c].astype("category")
    y = train[TARGET].to_numpy()

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for f, (tr, va) in enumerate(skf.split(X, y)):
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[va])[:, 1]
        scores.append(roc_auc_score(y[va], p))
        print(f"  fold {f}: {scores[-1]:.6f}")

    cv = float(np.mean(scores))
    diff = cv - RECORDED_SEED42_CV
    print(f"\nCV {cv:.6f} +/- {np.std(scores):.6f}")
    print(f"recorded in the ledger: {RECORDED_SEED42_CV:.6f}")
    print(f"difference: {diff:+.2e}  (expected: within {TOLERANCE:.0e})")
    print("reproduced" if abs(diff) < TOLERANCE else
          "OUTSIDE TOLERANCE - that would be worth investigating")
else:
    print(f"skipped. recorded CV for this config: {RECORDED_SEED42_CV:.6f}")

## Seed averaging, and where it stops paying

With `subsample=0.8` the model becomes stochastic, so the same config under a
different seed is a genuinely different model. Rank-averaging several of them is the
cheapest ensembling there is: no new ideas, no new features, just the same run again.

One warning first. **A single bagged run does not establish that bagging helps.**
Across five seeds the gain over the identical unbagged config was +0.000261,
+0.000024, +0.000235, +0.000127 and +0.000273. One seed got essentially nothing and
one finished *below* the unbagged model. Had I run one seed and read the result, I
would have believed whichever number I happened to draw. Only the blend is solid.

In [ ]:
sat = D["seed_saturation"]
n = [s["n_seeds"] for s in sat]
v = [s["cv"] for s in sat]

fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(n, v, color=BLUE, lw=2, marker="o", ms=8, mec=SURFACE, mew=1.5, zorder=3)
for xi, yi in zip(n, v):
    ax.annotate(f"{yi:.6f}", (xi, yi), textcoords="offset points", xytext=(0, 11),
                ha="center", fontsize=9, color=SECOND)
for i in range(1, len(n)):
    ax.annotate(f"+{(v[i] - v[i-1]) * 1e6:.0f}e-6",
                ((n[i] + n[i-1]) / 2, (v[i] + v[i-1]) / 2),
                textcoords="offset points", xytext=(0, -18), ha="center",
                fontsize=8.5, color=MUTED)
ax.set_xticks(n)
ax.set_xlabel("seeds in the rank-average blend")
ax.set_ylabel("cross-validation ROC AUC")
ax.set_ylim(min(v) - 0.00008, max(v) + 0.00012)
finish(ax, "Seed averaging saturates at four",
       "Each step is roughly half the last.")
plt.tight_layout()
plt.show()

Each step is about half the one before. Seeds six and seven would together buy
roughly 0.00003, so I stopped at five.

One negative result worth recording: **seed averaging did not reduce fold spread**,
0.000555 against 0.000549 for the best single model. The variance-reduction argument
for ensembling failed to show up three separate times in this competition. I have
stopped repeating it.

## The finding I did not expect: correlation does not predict blend value

The standard advice for ensembling is to look for models that are individually decent
and *uncorrelated*, because decorrelated errors cancel. I gated my whole diversity
plan on that, with thresholds written into a notebook before it ran.

So I measured it. Every pair of the twelve models I had saved out-of-fold predictions
for, 66 pairs, scored as a **50/50 rank blend**, against the Spearman correlation
between the two members. Gain is measured against **the better of the two members**,
which is the only comparison that answers "was blending worth it".

That "50/50" is doing more work than I realised at the time. Read this section and
the next one as measured, and then read the correction two sections down, because the
qualifier turns out to be the whole story.

In [ ]:
pairs = D["pairs"]
BAGGED = {"bagged seed 42", "bagged seed 2024", "bagged seed 7",
          "bagged seed 2025", "bagged seed 13"}
# The clean subset: pairs where the two members share an identical configuration
# and differ only in the random seed. Nothing varies but the stochasticity, so
# nothing is confounded with correlation.
homog = [p for p in pairs if p["a"] in BAGGED and p["b"] in BAGGED]
rest = [p for p in pairs if not (p["a"] in BAGGED and p["b"] in BAGGED)]

r = lambda xs, ys: float(np.corrcoef(xs, ys)[0, 1])
r_all = r([p["spearman"] for p in pairs], [p["gain"] for p in pairs])
r_hom = r([p["spearman"] for p in homog], [p["gain"] for p in homog])
r_gap = r([p["cv_gap"] for p in pairs], [p["gain"] for p in pairs])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.9))

ax = axes[0]
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
ax.plot([p["spearman"] for p in rest], [p["gain"] for p in rest], "o",
        color=MUTED, ms=6, alpha=0.55, lw=0, zorder=2, label="all other pairs")
ax.plot([p["spearman"] for p in homog], [p["gain"] for p in homog], "o",
        color=BLUE, ms=8, mec=SURFACE, mew=1.5, lw=0, zorder=3,
        label="identical config, seed only")
ax.set_xlabel("Spearman correlation between the two models")
ax.set_ylabel("blend gain over the better member")
ax.legend(frameon=False, loc="lower left", labelcolor=SECOND, fontsize=9)
finish(ax, "Correlation does not predict blend value",
       f"all 66 pairs r = {r_all:+.2f}   ·   the 10 clean pairs r = {r_hom:+.2f}")

ax = axes[1]
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
ax.plot([p["cv_gap"] for p in pairs], [p["gain"] for p in pairs], "o",
        color=BLUE, ms=6.5, mec=SURFACE, mew=1, alpha=0.9, lw=0, zorder=2)
ax.set_xlabel("difference in CV between the two models")
ax.set_ylabel("blend gain over the better member")
finish(ax, "Relative strength does, partly by definition",
       f"r = {r_gap:+.2f}, and a fixed 50/50 weight forces much of it")
plt.tight_layout()
plt.show()

print(f"spearman vs gain, all 66 pairs        : {r_all:+.3f}")
print(f"spearman vs gain, 10 seed-only pairs  : {r_hom:+.3f}")
print(f"cv gap vs gain,   all 66 pairs        : {r_gap:+.3f}")

**Left panel: correlation tells you essentially nothing.** Across all 66 pairs the
relationship between Spearman and blend gain is r = +0.14. The single best blend I
found came from one of the *most* correlated pairs.

**Right panel needs a caveat, and it is important.** Relative strength tracks blend
gain at r = -0.99, but a good part of that is definitional rather than discovered:
at a fixed 50/50 weight, averaging in a much weaker model *has* to drag the result
down. Read the right panel as a constraint the arithmetic imposes, not as a finding.

The honest test of the correlation claim is the blue points, ten pairs of models
with **identical configuration differing only in the random seed**, where nothing is
confounded with correlation. There, Spearman ranges over 0.990 to 0.996, blend gain
ranges over +0.000225 to +0.000360, and the relationship between them is **r =
+0.32 on ten points**, which is noise, and pointing the wrong way for the folk rule.

I want to be careful about what this does and does not license. It is one synthetic
tabular dataset, and every pair above is LightGBM against LightGBM. It is not a
claim that decorrelation is useless in general. What it does establish, for this
data, is that **Spearman was not a usable gate**, and the practical consequence is
concrete:

> CatBoost correlated with my LightGBM at 0.9877, squarely inside the 0.974 to
> 0.998 band that LightGBM produces against *itself*, and blended **negatively**, at
> -0.000250. The bagged seeds correlate *higher*, at 0.990 to 0.996, and blend
> **positively**, at +0.000546.

My original notebook would have read CatBoost's 0.9877 as "marginal diversity,
proceed" and spent the next day on a five-fold CatBoost run at nine times
LightGBM's cost, followed by XGBoost. Measuring the blend directly cost one fold and
stopped that.

**Gate on the measured blend AUC. It is one number and you can always afford it.**

That last line is right as far as it goes, and it is not far enough. Two sections
down it turns out that "the measured blend AUC" is not one number: it is one number
per combiner, and the answer changes sign depending on which one you pick.

## The one test that could have falsified it

Everything above is LightGBM against LightGBM, and that is a real weakness in
the argument. The 66 pairs span Spearman 0.975 to 0.998, and a folk rule about
decorrelation is not really on trial inside a band that narrow. CatBoost, the
one other family I had tried, landed at 0.9877, squarely inside it.

So I ran a genuinely different model: an embedding MLP,
512/256/128, quantile-transformed numeric inputs with an explicit missingness
mask, trained on a Kaggle T4 because the machine I work on has no GPU. Its
out-of-fold vector is the only one in this repo whose correlation with the
LightGBM blend falls **below** the within-family band.

If decorrelation pays, this is where it pays.

The fold split was checksummed on Kaggle against the local one before any of
this was blended. A misaligned out-of-fold vector blends perfectly cleanly and
is undetectable afterwards, which makes it the one error here worth a
dedicated check.


In [ ]:
nu = D["neural"]
curve = nu["weight_curve"]
lo, hi = nu["within_family_spearman_min"], nu["within_family_spearman_max"]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.9))

ax = axes[0]
ax.axvspan(lo, hi, color=BLUE, alpha=0.13, zorder=1)
ax.plot([p["spearman"] for p in D["pairs"]],
        np.full(len(D["pairs"]), 1.0), "|", color=BLUE, ms=18, mew=1.2,
        alpha=0.7, zorder=3)
ax.plot([nu["catboost_spearman"]], [1.0], "o", color=MUTED, ms=9,
        mec=SURFACE, mew=1.5, zorder=4)
ax.plot([nu["spearman_vs_blend"]], [1.0], "o", color=ORANGE, ms=10,
        mec=SURFACE, mew=1.5, zorder=5)
ax.annotate("the 66 within-family pairs", ((lo + hi) / 2, 1.0),
            textcoords="offset points", xytext=(0, 30), ha="center",
            fontsize=9, color=SECOND)
ax.annotate("CatBoost", (nu["catboost_spearman"], 1.0),
            textcoords="offset points", xytext=(0, -32), ha="center",
            fontsize=9, color=SECOND)
ax.annotate("neural", (nu["spearman_vs_blend"], 1.0),
            textcoords="offset points", xytext=(0, -32), ha="center",
            fontsize=9.5, color=ORANGE, fontweight="bold")
ax.set_ylim(0.5, 1.5)
ax.set_yticks([])
ax.grid(axis="y", visible=False)
ax.set_xlabel("Spearman correlation with the LightGBM blend")
finish(ax, "The only model that escapes the band",
       f"Within-family pairs run {lo:.4f} to {hi:.4f}. "
       f"The neural model sits at {nu['spearman_vs_blend']:.4f}.")

ax = axes[1]
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
ax.plot([c["w"] for c in curve], [c["gain"] for c in curve], color=ORANGE,
        lw=2, marker="o", ms=6, mec=SURFACE, mew=1.5, zorder=3)
ax.set_xlabel("weight given to the neural model")
ax.set_ylabel("blend gain over LightGBM alone")
won = sum(c["folds_won"] for c in curve)
finish(ax, "and the worst blend partner in the repo",
       f"Monotone down from the smallest weight tried. "
       f"{won} folds won out of {5 * len(curve)}.")
plt.tight_layout()
plt.show()

best = max(curve, key=lambda c: c["gain"])
print(f"neural CV         {nu['cv']:.6f} +/- {nu['sd']:.6f}")
print(f"LightGBM blend    {nu['lgbm_blend_cv']:.6f}")
print(f"behind by         {nu['cv'] - nu['lgbm_blend_cv']:+.6f}")
print(f"best weight w={best['w']:.2f}  gain {best['gain']:+.6f}, "
      f"wins {best['folds_won']}/5 folds")


**Spearman 0.9650, below every one of the 66 within-family pairs, and it is the worst
blend partner I measured.** The curve is monotone downward from the smallest weight I
tried. Zero folds won, at any weight. Nothing about it is marginal or close.

Every number in that chart is correct and it reproduces from the saved vector. What I
concluded from it was not. At the time I wrote: the decorrelated model did not help,
the near-identical bagged seeds did, and what predicts blend value on this data is not
how *different* two models are but whether they are **comparably strong**. Then I
wrote "the board is now closed" into my working notes and started writing this
notebook up.

## The one feature idea that worked, and it is not a feature

That closed board lasted six days. Two things came after it, and they are the two
largest results in the competition.

Three weeks of feature engineering on this dataset had produced nothing. Missingness
indicators: dead in ninety seconds, above. Threshold and rule features: below the
untuned baseline. Ratios and interactions: nothing. With twelve columns and a
generator that writes a smooth additive field, there was no hidden quantity to
construct.

Target encoding is not a new quantity. It replaces each column with the mean target
rate of the rows sharing its value, so a categorical or a discretised numeric arrives
as one number a tree can split on in a single cut, where before it cost many. Applied
to all twelve columns alongside a frequency encoding, 12 features become 36:

| | CV | fold sd |
|---|---|---|
| the same model, raw features | 0.963471 | 0.000591 |
| the 5-seed blend it replaced | 0.963880 | 0.000555 |
| **target encoded** | **0.966782** | **0.000453** |

**+0.003312 against the identical model on raw features, winning 5 folds out of 5,
with the per-fold differences having a standard deviation of 0.000270.** That is
twelve paired standard deviations. For scale, the entire seed-averaging programme
above was worth about one.

**The credit is not mine.** The idea came from a public notebook by tomasa2 on this
competition. The implementation, the leak checks and the cross-validation are mine,
which is the part worth writing down, because target encoding is the one thing in
this repo that can leak.

**How it is nested, and how I know it did not leak.** Inside each of the five outer
folds, an inner 5-fold split fits the encoder on four inner parts and writes the
encoded values for the fifth, so no row's own target ever reaches its own feature.
Smoothing 10 toward the prior. Three checks run inside the training notebook on the
full data rather than a sample:

- a validation row's encoded value against its own target: correlation **0.0**, since
  the encoder that produced it never saw that fold;
- a training row's encoded value against its own target: **8.1e-05**, below the
  **1.3e-04** that the leave-one-out arithmetic produces on its own;
- an encoder deliberately fit *without* the nesting, on the same folds: **0.9967**.

That third one is the check that matters. A leak here is worth three points of AUC,
so a broken implementation is loud. Reporting the two clean numbers without the third
would be reporting that an alarm did not go off without checking it was wired in.

### Two things that failed on top of it, both worth more than they cost

**Median-imputed copies of the nine numeric columns, added alongside the originals.**
A public notebook reported +0.0012 for this. I measured **+0.000023**, paired sd
0.000032. It won 4 folds of 5, which sounds positive until you notice the magnitude
is 0.7 paired standard deviations against target encoding's twelve. Fold count with no
magnitude behind it is not evidence. My guess, logged as a guess: they measured it
against a baseline without full target encoding, and once every column is encoded the
NaN level already receives its own value, so the imputed copies are redundant.

**The decimal lattice, and this is the one I would take away.** The float columns in
this generator are not uniform in their last digit, and the target rate depends on it.
Measured on the full training set, roughly 50,000 to 70,000 rows behind each digit:

| column | swing in target rate across the first decimal digit |
|---|---|
| `weekend_screen_time` | 11.54 points |
| `daily_screen_time_hours` | 8.90 points |
| `sleep_hours` | 7.12 points |
| `social_media_hours` | 5.53 points |

`daily_screen_time_hours` runs from 0.6482 at digit `.0` to 0.7373 at digit `.9`.
That is roughly 47 standard errors. The effect is real, it is enormous, and I verified
it here rather than taking it from the forum.

Adding the digit as a feature scored **-0.000132**, winning 1 fold of 5.

The reason is worth the paragraph. The digit is a **deterministic function of the
value**, so target encoding the value already carries whatever the digit contributes
to it. What a separate digit column adds is *pooling* across integer parts, and
pooling only pays when the per-value estimate is noisy. At about 500 rows per encoded
level, it is not. A large verified effect in the data can be worth exactly zero in
the model, and those are two different measurements.

## The correction: that was a fact about my combiner

The second thing that came after the closed board is a correction to the section
before last, and it is the one I would keep out of this whole notebook.

Every ensembling result above shares a hidden constant. The 66 pairs, the CatBoost
probe, the seven-way blend, the neural weight curve: **every one of them combined
models by averaging them.** Rank average, or a fixed share of one against the other.

A combiner with no weights can only average a member in. It cannot give it a small
weight and it cannot give it a negative one. So the experiment I never ran is the one
that holds the member set fixed and changes only the combiner.

Eighteen out-of-fold vectors: five target-encoded seeds, the twelve raw-feature
LightGBMs, and the neural model. Every stacker below is **fit on half the out-of-fold
rows and scored on the other half**, five random splits, paired. A stacker fitted and
scored on the same matrix reads high, and that optimism is exactly what would
manufacture the result I am claiming here, so it has to be measured out.

In [ ]:
comb = {c["name"]: c for c in D["combiners"]}
SHOW = ["rank mean, 5 seeds", "logit stack, 5 seeds",
        "rank mean, all 18", "logit stack, all 18"]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.9))

ax = axes[0]
vals = [comb[k]["gain"] for k in SHOW]
yy = np.arange(len(SHOW))
ax.barh(yy, vals, color=[ORANGE if v < 0 else BLUE for v in vals], height=0.62,
        zorder=3)
ax.axvline(0, color=AXIS, lw=1.2, zorder=4)
for i, v in enumerate(vals):
    ax.annotate(f"{v:+.6f}", (v, i), textcoords="offset points",
                xytext=(8 if v > 0 else -8, 0), va="center",
                ha="left" if v > 0 else "right", fontsize=9, color=SECOND)
ax.set_yticks(yy)
ax.set_yticklabels(SHOW, fontsize=9.5, color=SECOND)
ax.invert_yaxis()
ax.set_xlim(min(vals) * 1.55, max(vals) * 1.9)
ax.set_xlabel("gain over the best single model")
ax.grid(axis="y", visible=False)
finish(ax, "The combiner, not the members",
       "Bottom two rows are the same eighteen models. Averaging them loses;\n"
       "letting a logistic regression choose the weights wins.")

# Coefficients against each member's own CV. The claim in the title is that these two
# quantities come apart, so both have to be measured rather than asserted.
ax = axes[1]
co = D["stack_coefs"]
nn = [c for c in co if c["name"] == "neural"]
rest = [c for c in co if c["name"] != "neural"]
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
ax.plot([c["cv"] for c in rest], [c["coef"] for c in rest], "o", color=MUTED, ms=7,
        alpha=0.75, lw=0, zorder=2, label="LightGBM, 17 of them")
ax.plot([c["cv"] for c in nn], [c["coef"] for c in nn], "o", color=ORANGE, ms=10,
        mec=SURFACE, mew=1.5, lw=0, zorder=3, label="the neural model")
ax.annotate("rejected on the evidence\nin the section above",
            (nn[0]["cv"], nn[0]["coef"]), textcoords="offset points", xytext=(12, -6),
            fontsize=9, color=SECOND, linespacing=1.35)
ax.set_xlabel("the member's own cross-validation AUC")
ax.set_ylabel("its weight in the stack")
ax.legend(frameon=False, loc="lower right", labelcolor=SECOND, fontsize=9)
finish(ax, "Weight is not strength",
       "The weakest member of the eighteen takes the seventh largest weight.\n"
       "The negative weights are underfit models used as corrections.")
plt.tight_layout()
plt.show()

for k in SHOW:
    c = comb[k]
    print(f"{k:22} {c['auc']:.6f}  {c['gain']:+.6f}  "
          f"sd {c['paired_sd']:.6f}  {c['splits_won']}/5 splits")

**Same eighteen models. Opposite sign.** Averaged, they lose 0.001554 and win zero
splits out of five. Weighted, they gain 0.000908 and win all five, with a paired
standard deviation of 0.000018. The distance between those two bars is comparable to
the entire target-encoding result, and none of it is attributable to the models.

Both of those are split-half numbers. Refitting the combiner inside the five-fold loop
instead, which is how every other number in my ledger is produced, gives **+0.000867
on five folds out of five**. The result does not depend on which of the two honest
protocols measures it, and the optimism in the naive version, fitting and scoring the
stacker on the same rows, turned out to be 1.5e-05: eighteen parameters on 691,369
rows is not enough to overfit with.

The two upper bars are the other half of the story. Among five seeds of one
configuration, the rank average and the fitted stack agree to five decimals. When
members are equally strong and near identical, a stacker has nothing to do that an
average is not already doing. **That is the regime every ensembling experiment in this
repo was run in**, which is why those results were trustworthy and why the conclusion
I drew from them was not.

The neural model is the sharpest case, since it is the one I closed the board on:

| paired with the best single model | gain | splits won |
|---|---|---|
| rank average | -0.006272 | 0/5 |
| logistic stack | **+0.000090** | **5/5** |

Same two vectors, same rows. Averaged it is a disaster and the weight curve above was
right. Weighted it is positive on every split. Dropping it from the eighteen and
refitting the other seventeen costs **0.000093**, paired sd 0.000003, five splits out
of five: a tenth of the stack's total gain, from the model I had called the worst
partner I measured.

The right panel says why. `anchor` and `300 trees` are underfit LightGBMs whose errors
are systematic, so the stacker takes them at **-0.36** and **-0.13** and uses them as
corrections. `bagged seed 2024` is a strong model nearly identical to four others in
the set and gets **+0.01**, because it is redundant. The neural model gets **+0.12**
because it is the only member wrong in a different direction.

**What survives, stated carefully.** Rank correlation still does not predict
equal-weight blend value; the 66-pair result is untouched, and so is the CatBoost
decision, which was made on cost as well. Averaging in a much weaker model still
drags the average down, which is arithmetic. What does not survive is the
generalisation I made from those: that a weak decorrelated model has no value on this
data. It has value the moment the combiner is allowed to weight it.

**How I missed it for a week.** I had already caught myself once here. The original
gate was the Spearman threshold, I measured that it was anti-predictive, and I
replaced it with "measure the blend AUC directly". That was the right correction and
it fixed the statistic. It left the combiner exactly where it was, and by then the
answer looked known, so nothing asked the next question. The cost was one rejected
model, three days of a closed board, and most of a writeup arguing the wrong
conclusion at length.

**The rule I would give someone else:** a diversity claim is a claim about a member
set *and* a combiner. Put both in the sentence, or you will generalise one of them by
accident.

## The correction had a second half, and I did not see it for ten days

The section above fixed my combiner. What it did not do is go back and ask which
conclusions had been reached with the broken one and were still standing.

Two had. CatBoost and the neural model were both rejected, and both rejections were
made under the equal-weight combiner **and** on the raw feature set, before target
encoding existed. Fixing the combiner did not retract either of them, because nothing
went back to look.

There is a distinction I had collapsed, and it is the part of this notebook I expect to
reuse: **"rejected" and "never run under the current feature set" are different
statuses.** The first is a result. The second is the absence of one, wearing a result's
clothes.

So I re-ran both, one variable at a time, against the model each had originally lost to.


In [ ]:
r = D["reopenings"]
cat, nn = r["catboost"], r["neural"]
lgb_seeds = sorted(D["target_encoding"]["seed_cv"].values())
cat_seeds = sorted(cat["seed_cv"].values())

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.9))

# Left: five seeds of each, same folds, same encoder, same budget. Both the level and
# the spread are the point, so the seeds are drawn individually rather than as a mean.
ax = axes[0]
for i, (vals, colour, lbl) in enumerate([(lgb_seeds, MUTED, "LightGBM"),
                                         (cat_seeds, BLUE, "CatBoost")]):
    ax.plot(vals, [i] * len(vals), "o", color=colour, ms=9, alpha=0.85, lw=0, zorder=3)
    m = sum(vals) / len(vals)
    ax.plot([m], [i], "|", color=INK, ms=26, mew=2, zorder=4)
    ax.annotate(f"{lbl}   spread {max(vals) - min(vals):.1e}", (m, i),
                textcoords="offset points", xytext=(0, 17), ha="center",
                fontsize=9.5, color=SECOND)
ax.set_yticks([0, 1])
ax.set_yticklabels([])
ax.set_ylim(-0.6, 1.7)
ax.set_xlabel("cross-validation AUC, five model seeds each")
ax.grid(axis="y", visible=False)
finish(ax, "CatBoost, on the features it was never tried on",
       f"Family means differ by {cat['gap']:+.6f}, which is "
       f"{cat['gap_in_se']:.1f} standard errors.\n"
       f"On the raw features it had lost by 0.001675.")

# Right: the marginal value of each member set, fit inside the fold loop so every
# number is comparable to a single model's CV.
ax = axes[1]
cur = D["stack_curve"]
xs = [c["n"] for c in cur]
ys = [c["cv"] for c in cur]
ax.plot(xs, ys, "-o", color=BLUE, ms=8, lw=2, zorder=3)
for c in cur:
    if c["step"]:
        ax.annotate(f"{c['step']['gain']:+.6f}", (c["n"], c["cv"]),
                    textcoords="offset points", xytext=(-6, 10), ha="right",
                    fontsize=9, color=SECOND)
ax.annotate("eighteen members,\nwhere the section above ended", (xs[0], ys[0]),
            textcoords="offset points", xytext=(14, -4), fontsize=9, color=SECOND,
            linespacing=1.35)
ax.set_xticks(xs)
ax.set_xlabel("members in the stack")
ax.set_ylabel("stack CV, combiner fit inside the fold loop")
finish(ax, "Every addition after the first is worth less",
       "The step labels are the marginal gain from that addition,\n"
       "paired per fold and winning five folds out of five in each case.")

plt.tight_layout()
plt.show()


**CatBoost had lost by 0.001675 and now leads by 0.000157.** The old number is a single
fold from the original probe and I am quoting it as one, so it is not directly
comparable to the five-fold figures beside it. The new one is five seeds against five
seeds on identical folds, and the gap between the two family means is 13 standard
errors, which is not a close call.

The second thing in that left panel is one I was not looking for. **CatBoost's seed
spread is about five times tighter than LightGBM's**, 1.3e-05 against 6.1e-05. That
continues a pattern rather than starting one: on the raw features my LightGBM seeds
spanned 2.5e-04, target encoding took them to 6.1e-05, and CatBoost on those same
encoded features sits at 1.3e-05. Every step toward the representation the model wanted
also made it less sensitive to its own randomness. I have no mechanism for that and I
am recording it without one.

Then the neural model, which is the one I had closed the board on twice:

| the same architecture, folds, seed and early stopping | CV |
|---|---|
| on the raw twelve features | 0.939169 |
| **on the encoded thirty-six** | **0.965373** |

**+0.026204, five folds out of five.** It is the largest single-model gain in the whole
competition, and it came from a model I had written up as the closing argument for a
conclusion that was itself wrong.

That jump is +2.79%, and my own rules say to treat anything above 2% as a leak until
proven otherwise. So: it is not one, and the argument is a ceiling rather than a clean
bill of health. The new number lands **below** both gradient boosters on the identical
encoder, and an encoder with the nesting deliberately removed scores 0.9967 on these
folds, so a leak here is loud. The rule fired because the baseline was weak, not
because the new number is out of reach.


In [ ]:
# The 24-member stack. The question is whether the strong neural model simply replaces
# the weak one, which would mean they were one mechanism all along.
co = D["stack24_coefs"]
lookup = {c["name"]: c["coef"] for c in co}
fam = lambda n: ("neural_te" if n == "neural_te" else
                 "neural" if n == "neural" else
                 "catboost" if n.startswith("cat") else "lightgbm")
STYLE = {"lightgbm": (MUTED, 6, "LightGBM, 18 of them"),
         "catboost": (BLUE, 7, "CatBoost, five seeds"),
         "neural": (ORANGE, 10, "the raw-feature neural model"),
         "neural_te": (INK, 11, "the encoded neural model")}

fig, ax = plt.subplots(figsize=(8.4, 5.2))
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
for key, (colour, ms, lbl) in STYLE.items():
    pts = [c for c in co if fam(c["name"]) == key]
    ax.plot([c["cv"] for c in pts], [c["coef"] for c in pts], "o", color=colour,
            ms=ms, mec=SURFACE, mew=1.2 if ms > 7 else 0, lw=0, zorder=3, label=lbl)

ax.annotate(f"largest weight in the stack\n{lookup['neural_te']:+.4f}",
            (next(c["cv"] for c in co if c["name"] == "neural_te"),
             lookup["neural_te"]),
            textcoords="offset points", xytext=(-14, -30), ha="right", fontsize=9,
            color=SECOND, linespacing=1.35)
ax.annotate(f"keeps {lookup['neural'] / 0.1181:.0%} of the weight\n"
            f"it had before its replacement arrived",
            (next(c["cv"] for c in co if c["name"] == "neural"), lookup["neural"]),
            textcoords="offset points", xytext=(16, -4), fontsize=9, color=SECOND,
            linespacing=1.35)
ax.set_xlabel("the member's own cross-validation AUC")
ax.set_ylabel("its weight in the 24-member stack")
ax.legend(frameon=False, loc="lower right", labelcolor=SECOND, fontsize=9)
finish(ax, "Two neural models, both paid",
       "If they were the same mechanism at two strengths, the weak one\n"
       "would have collapsed when the strong one joined. It did not.")
plt.tight_layout()
plt.show()


**The encoded neural model takes the largest weight in the stack**, above every
gradient booster in it. And the weak one does not collapse: it keeps most of the weight
it had before its own replacement arrived. They are wrong in different directions from
each other, not one mechanism at two strengths.

Now the part that keeps this honest, and it is the number I would lead with if I were
reading this rather than writing it.

**A 0.026 single-model gain was worth 0.000043 to the stack.** That is the last step in
the right-hand panel above. The largest single-model improvement of the competition
moved the thing I actually submit by about four hundredths of what it moved the model.

That is not a contradiction of the correction. It is the same fact from the other side.
A fitted combiner can extract most of a direction's value from a weak member, which is
exactly why the weak neural model was worth keeping at +0.118 and why dropping it cost
a tenth of the stack's gain. The identical property means that once the direction is
already represented, making its representative better adds very little. Both halves
come from the combiner being able to weight.

So the correction earns back two rejected models and one genuinely large single-model
result, and buys **0.000157 of stack CV** across all of it. On a leaderboard where the
top fifty sit 0.0021 above me, that is honest bookkeeping rather than a comeback. What
it changed is what I now believe about my own process, which was worth more than the
score.


## The third reopening: the model I rejected without ever running it

The two above were models I had run and beaten. This one is worse, and it is the
reason I now separate the two statuses in writing.

Here is what I wrote in my own notes, in the list of things I was deliberately not
going to do:

> **XGBoost.** A third histogram GBDT on the same 12 features. The CatBoost result
> removes the reason to expect it to behave differently.

That is a conclusion resting entirely on a premise. The premise was that CatBoost had
lost. Two sections ago the premise was withdrawn, and the conclusion sat there for
another week with nothing under it, in a list of things I had decided not to do, which
is not a place anyone goes looking for assumptions to re-check.

So: same encoder, same folds, same seed, same budget as the LightGBM it was supposed to
resemble. One variable.


In [ ]:
r = D["reopenings"]
xg, cat = r["xgboost"], r["catboost"]
lgb_seeds = sorted(D["target_encoding"]["seed_cv"].values())
cat_seeds = sorted(cat["seed_cv"].values())
xgb_seeds = sorted(xg["seed_cv"].values())

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.0))

# Left: three families, five seeds each, identical folds and encoder. Drawn as
# individual seeds because the spread is half the point.
ax = axes[0]
FAMS = [(lgb_seeds, MUTED, "LightGBM"), (cat_seeds, BLUE, "CatBoost"),
        (xgb_seeds, ORANGE, "XGBoost")]
for i, (vals, colour, lbl) in enumerate(FAMS):
    ax.plot(vals, [i] * len(vals), "o", color=colour, ms=9, alpha=0.85, lw=0, zorder=3)
    m = sum(vals) / len(vals)
    ax.plot([m], [i], "|", color=INK, ms=26, mew=2, zorder=4)
    ax.annotate(f"{lbl}   mean {m:.6f}", (m, i), textcoords="offset points",
                xytext=(0, 17), ha="center", fontsize=9.5, color=SECOND)
ax.set_yticks(range(len(FAMS)))
ax.set_yticklabels([])
ax.set_ylim(-0.6, 2.7)
ax.set_xlabel("cross-validation AUC, five model seeds each")
ax.grid(axis="y", visible=False)
finish(ax, "Three gradient boosters on the encoded features",
       f"XGBoost leads LightGBM by {xg['gap']:+.6f}, which is "
       f"{xg['gap_in_se']:.1f} standard errors of the\ntwo-sample difference, and "
       f"leads CatBoost by {xg['gap_vs_catboost']:+.6f}.")

# Right: what each reopening was worth to the stack, which is the only currency the
# three are comparable in. The two null sweeps are drawn at zero on purpose.
ax = axes[1]
step = {c["n"]: c["step"] for c in D["stack_curve"] if c["step"]}
BARS = [("CatBoost", step[19]["gain"], BLUE),
        ("neural, on the\nencoded features", step[24]["gain"], BLUE),
        ("XGBoost", step[25]["gain"], BLUE),
        ("num_leaves", 0.0, MUTED),
        ("encoder\nsmoothing", 0.0, MUTED)]
xs = range(len(BARS))
ax.bar(xs, [b[1] for b in BARS], color=[b[2] for b in BARS], width=0.6, zorder=3)
ax.axhline(0, color=AXIS, lw=1.2, zorder=2)
for i, (lbl, v, _) in enumerate(BARS):
    if v:
        ax.annotate(f"{v:+.6f}", (i, v), textcoords="offset points", xytext=(0, 5),
                    ha="center", fontsize=9, color=SECOND)
    else:
        ax.annotate("null", (i, 0), textcoords="offset points", xytext=(0, 6),
                    ha="center", fontsize=9.5, color=SECOND)
ax.set_xticks(list(xs))
ax.set_xticklabels([b[0] for b in BARS], fontsize=9, color=SECOND, linespacing=1.3)
ax.set_ylim(0, step[19]["gain"] * 1.28)
# The distinction the panel is about goes in a legend, not in five tick labels.
ax.bar([0], [0], color=BLUE, label="a different learner")
ax.bar([0], [0], color=MUTED, label="a hyperparameter, best arm of the sweep")
ax.legend(frameon=False, loc="upper right", labelcolor=SECOND, fontsize=9)
ax.set_ylabel("gain in stack CV from adding it")
finish(ax, "Learners paid, knobs did not",
       "Every bar is a paired five-fold gain on the same out-of-fold matrix,\n"
       "winning five folds out of five wherever it is non-zero.")

plt.tight_layout()
plt.show()


**XGBoost is the best single model in the competition for me**, at a family mean of
0.967118 against LightGBM's 0.966763. The gap is +0.000355 at 23.6 standard errors of
the two-sample difference, and it beats the CatBoost I had just spent a week reinstating
by a further +0.000197.

Two things about that number are worth saying out loud.

**It is family against family, and I got that wrong the first time.** I originally
logged the gap as +0.000336 at 35.1 standard errors, which compared one family mean
against a single seed's error bar. The correct comparison pools both families and gives
a wider interval on a slightly larger difference. It was caught because I compute these
numbers in two places, and the two disagreed. A duplicated calculation is not waste; it
is the only reason that error is in this notebook as a correction rather than in it as
a fact.

**One seed would not have been enough.** Row 38 was a single seed at +0.000316, which
is inside what a lucky seed can produce in this repo. It only became a result after
four more seeds, and that is the standing rule here: CatBoost and XGBoost both had to
clear it and both did.

### It paid for itself out of the LightGBM seeds, not out of the stack

The more interesting result is not the level, it is what happened to the combiner when
XGBoost joined. This is the substitution, and it is unusually clean.


In [ ]:
# Same members, one addition. The question is where the new weight comes from.
was = {c["name"]: c["coef"] for c in D["stack24_coefs"]}
now = {c["name"]: c["coef"] for c in D["stack29_coefs"]}
te = sorted([n for n in was if n.startswith("te")], key=lambda n: -was[n])
xgb = sorted([n for n in now if n.startswith("xgb")], key=lambda n: -now[n])

te_was, te_now = sum(was[n] for n in te), sum(now[n] for n in te)
xgb_now = sum(now[n] for n in xgb)

fig, ax = plt.subplots(figsize=(8.8, 5.0))
ax.axvline(0, color=AXIS, lw=1.2, zorder=1)

# Dumbbells: where each LightGBM target-encoded seed sat before, and after.
for i, n in enumerate(te):
    ax.plot([was[n], now[n]], [i, i], color=GRID, lw=2.5, zorder=2)
    ax.plot([was[n]], [i], "o", color=MUTED, ms=9, zorder=3)
    ax.plot([now[n]], [i], "o", color=ORANGE, ms=9, mec=SURFACE, mew=1.2, zorder=4)
# The five XGBoost seeds have no "before", so they are drawn as arrivals.
for j, n in enumerate(xgb):
    i = len(te) + 0.6 + j
    ax.plot([0, now[n]], [i, i], color=GRID, lw=2.5, zorder=2)
    ax.plot([now[n]], [i], "D", color=BLUE, ms=8, mec=SURFACE, mew=1.2, zorder=4)

ax.set_yticks(list(range(len(te))) + [len(te) + 0.6 + j for j in range(len(xgb))])
ax.set_yticklabels(te + xgb, fontsize=9)
ax.set_xlabel("weight in the logistic stack")
ax.grid(axis="y", visible=False)
ax.annotate(f"five LightGBM TE seeds\n{te_was:+.4f} together, then {te_now:+.4f}",
            (was[te[0]], 0), textcoords="offset points", xytext=(10, 14),
            fontsize=9, color=SECOND, linespacing=1.35)
# Placed in the empty upper-left block rather than beside a marker, which is where
# the arrivals leave a gap.
ax.text(0.004, len(te) + 0.6 + len(xgb) - 1,
        f"five XGBoost seeds arrive\nat {xgb_now:+.4f} together",
        fontsize=9, color=SECOND, linespacing=1.35, va="center")
finish(ax, "XGBoost did not add to the stack, it replaced part of it",
       "Grey is the 24-member fit, colour is the 29-member fit.\n"
       "Same rows, same combiner, same regularisation. Only the member set moves.")
plt.tight_layout()
plt.show()

print(f"five LightGBM TE seeds, summed weight: {te_was:+.4f} -> {te_now:+.4f}")
print(f"five XGBoost seeds, summed weight:     {xgb_now:+.4f}")
print(f"stack CV: {D['stack_curve'][3]['cv']:.6f} -> {D['stack_curve'][-1]['cv']:.6f}")


**The five LightGBM target-encoded seeds go from +0.386 of weight between them to
+0.006.** They are not downweighted, they are switched off. XGBoost arrives carrying
+0.464 and the combiner takes almost exactly that much back out of the models it
replaced.

This is substitution, and it says something the CV column does not. Those five models
were never contributing five models' worth of information. They were one direction,
represented five times, and the moment a better representative of that direction turned
up the combiner stopped paying for them. The stack went up by only +0.000119 across both
XGBoost steps precisely because most of what XGBoost knows was already in there.

I then did the obvious follow-up and it is the one experiment in this competition that
tries to make the model *smaller*: drop those five members and refit. It costs **four
millionths**, which is nothing, on a stack of 29 members losing five of them.

**And I am not acting on it.** The five were selected by reading the coefficients off
the same out-of-fold matrix the result is then measured on, which is selection on the
validation set, and I had pre-registered a check that any drop be justified by
training-fold information alone. The check failed. It also turned out to be
mis-specified, comparing a maximum over folds against a mean over folds, so it was
close to guaranteed to fail whatever the data did. That is my error and it is in the
ledger as one.

The verdict stands as written anyway. Moving a pre-registered bar after seeing the
number is the exact failure the bar exists to prevent, and a bar that only binds when
it agrees with you is not a bar. The substantive conclusion survives without it: the
selection bias here can only *inflate* the pruned stack, since the members were chosen
for looking useless on these very rows, and the inflated estimate is still +0.000004.


## The rule that came out of the three, and the one prediction I made in advance

Three reopenings after the same event, all three paying, is a pattern worth trying to
state. So I stated it, and then I got two chances to be wrong about it.

After target encoding landed I also had three hyperparameter knobs that had never been
searched. `num_leaves` had sat at LightGBM's default of 31 for the entire competition
and appeared in no notebook. The encoder's own smoothing constant and its inner split
count had both been set on the day the encoder was written, chosen once, and inherited
by 22 of the 29 models in my best stack. All three are exactly the kind of thing that
looks negligent in a writeup.

All three returned nothing.

| reopened | what changed | result |
|---|---|---|
| CatBoost | the learner | **+0.000157** as a family |
| the neural model | the learner's input representation | **+0.026204** single-model |
| XGBoost | the learner | **+0.000355** as a family |
| `num_leaves`, swept 15 to 127 | a hyperparameter | null, 31 was already right |
| encoder smoothing, swept 1 to 100 | a hyperparameter | null, flat below 10 |
| encoder inner splits, swept 2 to 20 | a hyperparameter | null, flat above 5 |

**A change of representation revalues learners, not their knobs.** Which reads as
obvious written down, and was not obvious at all while I was deciding what to spend a
week of compute on.

The mechanism I would offer is that a learner and a representation are matched to each
other, so changing one genuinely re-runs the competition between learners. A knob is
fitted downstream of both. If it was near-optimal on the old representation it is
usually near-optimal on the new one, because the thing it is optimising against did not
move much.

The part that makes this worth including rather than a rationalisation: **the rule was
written down after four of those six rows and it called the last two in advance.** Both
sweeps were predicted null in their notebook headers, from the arithmetic, before the
runs. A rule invented after the last data point explains everything and predicts
nothing, and I would not be showing you this table if the fifth and sixth rows had been
found first.

The sixth is the better test, because that header did not only predict a null. It also
wrote down the strongest argument *against* the rule and made it falsifiable. The inner
split count controls a real train/serve mismatch: training rows get their encoding from
a fraction of the fold while validation rows get theirs from all of it, and raising the
count narrows that gap. That predicts a curve rising toward the top of the grid. It does
not rise. The notebook prints `monotone rising: False` from a check written before the
run, and the mismatch turns out to be real, measurable in the encoded columns, and worth
nothing in the model.

I am reporting that at the same volume as the null itself, because a header that only
mentions its predictions when they land is not making predictions.

It also carries a real cost that I should not hide. The rule says XGBoost was worth
running and `num_leaves` was not. It would have said the same thing about the rejections
I sat on for a week, so what it actually buys is a *reason to look*, not a new capability.
I had all the information needed to run XGBoost ten days before I ran it.

### The positive control, which is the part I would steal from this

A sweep whose predicted result is "no difference" has a failure mode that looks
identical to success. If the constant had never reached the encoder at all, every arm
would have been a copy, the curve would have come back perfectly flat, and I would have
written the same conclusion from a bug.

So the smoothing notebook builds one fold at two very different values and compares the
encoded columns directly, before any model runs. They differ by 0.366, so the constant
is live and the flat curve means what it says. It is four lines and it is the only
reason that null is worth anything.

The inner-split sweep sharpened this, and it is the version I would actually reuse. That
knob is read by one pass of the encoder and no other, so it has a side it must **not**
reach: training-row encodings have to move between arms, and validation-row encodings
have to stay identical to the bit. Measured before any model trained, the training
columns moved by 0.164 and the validation columns moved by exactly zero.

**A one-sided control only catches the knob that does nothing.** The knob that does too
much is the more dangerous of the two, because it does not produce a suspiciously flat
curve. It produces a perfectly plausible one, for a different experiment than the one
you think you are running.

Two constants, two nulls, and the pipeline now has nothing in it that was set once and
never checked. That is not a leaderboard result. It is the thing that makes me willing
to put the leaderboard result in writing.


## Fold standard deviation is the wrong bar for a paired comparison

A rule I had written down for myself, and which is good general advice: do not
believe an improvement smaller than the fold-to-fold spread.

By that rule my one real ensembling result was inconclusive. The seed blend gained
**+0.000546** over the previous best against a fold spread of **0.000549**.

The rule is wrong here, and it is worth understanding why. The per-fold differences
were:

```
+0.000588  +0.000631  +0.000476  +0.000620  +0.000419
```

Five folds out of five, with a standard deviation across those differences of
**0.000094**, a sixth of the fold spread.

Fold-to-fold variation is mostly driven by *which rows landed in which fold*. That
variation is common to both models being compared, so it **cancels in the
difference**. Comparing a paired improvement against the unpaired fold spread throws
away the pairing and inflates your bar by roughly six times.

**The right bar for "is model B better than model A" is the spread of the per-fold
differences, plus how many folds it wins.** By that bar the result is unambiguous.
It nearly got discarded.

## What CV and the public leaderboard actually did

In [ ]:
# Row 16 is the neural model at 0.9392. Different family, never submitted, and
# plotting it here would flatten every other row into a line. It has its own figure
# above rather than being dropped in silence, and the title below says so.
led = [r for r in D["ledger"] if r["id"] != 16]
x = [r["id"] for r in led]
cv = [r["cv_mean"] for r in led]
by_id = {r["id"]: r["cv_mean"] for r in led}
# `is not None` matters: an unsubmitted row carries a null here, and a bare
# truthiness test would let it through and rely on matplotlib silently dropping it.
lb_x = [r["id"] for r in led if r["lb_public"] is not None]
lb_y = [r["lb_public"] for r in led if r["lb_public"] is not None]

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.plot(x, cv, color=BLUE, lw=2, marker="o", ms=5.5, label="cross-validation",
        zorder=3, mec=SURFACE, mew=1.5)
ax.plot(lb_x, lb_y, color=ORANGE, lw=0, marker="D", ms=7, mec=SURFACE, mew=1.5,
        label="public leaderboard", zorder=4)
for k, (xi, yi) in enumerate(zip(lb_x, lb_y)):
    crowded = k and xi - lb_x[k - 1] < 4
    ax.annotate(f"{yi:.5f}", (xi, yi), textcoords="offset points",
                xytext=(0, -17 if crowded else 9), ha="center", fontsize=8.5,
                color=SECOND)
ax.annotate("untuned anchor", (1, by_id[1]), textcoords="offset points",
            xytext=(12, -13), fontsize=9, color=SECOND)
ax.annotate("target encoding", (17, by_id[17]), textcoords="offset points",
            xytext=(4, -22), fontsize=9, color=SECOND)
ax.annotate("XGBoost", (38, by_id[38]), textcoords="offset points",
            xytext=(0, -21), ha="center", fontsize=9, color=SECOND)
ax.annotate("the 29-member stack", (44, by_id[44]), textcoords="offset points",
            xytext=(-4, 12), ha="right", fontsize=9, color=SECOND)
# The tail is jagged because rows 35-37 and 46-49 are sweep arms, deliberately
# run below the best model to find where a curve stops being flat. They are
# experiments, not attempts, and they belong on this chart for that reason.
ax.set_xlabel("experiment")
ax.set_ylabel("ROC AUC")
ax.set_xticks([n for n in x if n % 2 == 0])
ax.legend(frameon=False, loc="lower right", labelcolor=SECOND)
finish(ax, "Every experiment in the ledger",
       "Fifteen experiments of capacity and averaging, worth +0.0089 together.\n"
       "Then one change of representation, worth +0.0029, and three reopenings\n"
       "it made possible worth +0.0006 more on top of the stack.")
plt.tight_layout()
plt.show()


The leaderboard sits above CV by between 0.0010 and 0.0022 on every one of the nine
submissions, and **seven of the eight consecutive comparisons agree in direction**.
That is the whole job of a validation scheme, and it is worth stating as a fraction
rather than as a slogan, because the one exception is the most informative submission
pair I have.

The strongest version of the check is the four submissions that changed the feature
set or the member set rather than a hyperparameter: CV moved +0.0029, +0.0009, +0.0001
and +0.0002, and the leaderboard moved +0.0032, +0.0007, +0.0001 and +0.0002. Row 24
was predicted at 0.9691 from the running offset before it was submitted and came back
at 0.96897.

Now the exception, which is the two submissions early on. My 3-seed and 5-seed blends are
separated **cleanly** by CV: the 5-seed wins 5 folds out of 5 with a paired standard
deviation of 0.000023. On the public leaderboard they scored **0.965090** and
**0.965080**, a gap of 0.00001 in the other direction.

That is not a CV/LB disagreement to diagnose. The public leaderboard here is a sample
of roughly 89,000 rows and its standard error is near **0.001**, larger than every
single gain after the fourth experiment. It is a 6e-5 effect measured with a 1e-3
ruler.

The practical consequence for final submission selection: the usual split is best-CV
plus best-public-LB, and when the public LB's noise floor exceeds the effects you are
choosing between, **best-public-LB is close to picking at random**. CV gets the
heavier weight here.

One more thing I got wrong, and it is embarrassing in a useful way. Three stacks in
the middle of that sequence were never submitted, because I had decided a submission
slot was scarce and each one was too close to its predecessor to be worth spending
one on. The daily limit is ten. I had never checked, and I was comparing each stack to
the previous stack rather than to the submission actually sitting on the leaderboard,
which is the comparison that decides whether a slot buys information.


## The ledger, and what I would do differently

Every experiment is written down, including the failures. The failures are most of the
value: they are why the same dead end did not get walked twice, and several of them are
in this notebook at the same length as the wins.

**What worked**

- Fitting model capacity first. 80% of the gain from the first three weeks.
- Lower learning rate with a compensating tree count. A small gain, and a useful side
  effect: fold spread fell from 0.000816 to 0.000549, so every later comparison got
  easier to call.
- Bagged seed averaging, rank-blended, up to four seeds.
- Target and frequency encoding on all twelve columns, +0.0029, nested inside the fold
  loop and leak-checked three ways. The largest single decision in the competition, and
  the one that made everything below it possible.
- A logistic stack over everything trained, including the models that were individually
  rejected.
- Reopening three rejected models after the representation changed. CatBoost, the
  neural model, and XGBoost, which had never been run at all and turned out to be the
  best single model here.

**What did not**

- Missingness indicators. No signal, established in ninety seconds.
- Threshold and rule features. The generator caps them below the untuned baseline.
- External data. The original is 7,500 rows against 691,369, its distributions are
  warped relative to the synthetic, and it appears to be synthetic itself.
- Capacity diversity within LightGBM, under an equal-weight blend.
- Median-imputed columns on top of target encoding. +0.000023, a null.
- The decimal lattice. An 11-point effect in the data, -0.000132 in the model.
- `num_leaves`. Never searched until the last week, and 31 was already right.
- The encoder's smoothing constant. Set once and never varied, and flat below 10.
- The encoder's inner split count. Set once on the same day, and flat above 5. With it
  the pipeline has no constant left that was chosen once and never checked.
- Pruning five members out of the stack. It costs four millionths, and the
  pre-registered check said do not act on it, so I did not.

**What I got wrong and had to publish a correction to**

- The neural model, and with it the whole diversity conclusion. It is 0.0247 behind and
  it makes an *average* worse at every weight, which is what I measured and what I
  reported. What I then claimed was that diversity does not pay on this data, and that
  was a fact about equal-weight combiners rather than about the data. It takes a
  positive weight in the stack, and dropping it costs a tenth of the stack's gain.
- CatBoost, which I wrote off on a single fold of the raw features and which leads
  LightGBM by 13 standard errors on the encoded ones.
- The XGBoost gap, reported first as one family against one seed's error bar and
  corrected to family against family. Caught only because the number is computed in two
  places and they disagreed.

**What I would do differently**

1. **Time one fold before launching anything.** I started a CatBoost run estimated at
   30-50 minutes. It ran for two hours with `verbose=0` and no progress output before
   being killed. It was never hung: a single fold was ten minutes and the notebook
   called the pipeline twice. The estimate was wrong by 3x because no fold had been
   timed.
2. **Name the combiner in any claim about ensembling.** See above. This is the
   expensive one.
3. **Keep a list of rejections with the reason attached, and re-read it whenever the
   reason changes.** Every rejection in this competition was correct when it was made.
   Three of them stopped being correct on the same afternoon and none of them moved,
   because a decision, once filed, does not announce that its premise has expired. This
   is the cheapest thing on the list and it was worth the most.
4. **Not offer a mechanism from one data point.** I attributed the CV/LB gap to fold
   averaging on the first submission, patched the story on the second, and withdrew it
   on the third when the pattern did not hold.
5. **Check the constraint before optimising against it.** I held back three submissions
   to conserve a daily quota I had never looked up. It is ten.
6. **Not declare a board closed.** I wrote that sentence with eight rejected ideas
   behind it, and the two largest gains in the competition came after it. Then I wrote
   it again, and XGBoost came after that.

**What I would keep**

The checksummed fold split. It is ten lines, and it is the reason an out-of-fold vector
saved in week one could be stacked against one from week four with any confidence at
all. Half of this notebook would be unmeasurable without it.

And the habit of writing the predicted result into the notebook header before the run.
It costs nothing, and it is the only thing that separates a rule with six points on it
from a story told about six points.

---

Ledger, notebooks and working log: **github.com/vyask21/smartphone-addiction**

If one thing here is worth taking away, it is that a cheap test of a wrong idea is
worth more than an expensive test of a good one, and most ideas are wrong. Including,
three times in this notebook, mine.
